In [ ]:
module load apps/binapps/anaconda3/2022.10
micromamba env create -f scrna_env.yaml -y
conda activate scrna

In [ ]:
# ✅ 新代码（无任何服务器信息，可直接传GitHub）
import scanpy as sc
import os
from config import DATA_RAW  # 导入统一配置

# 读取数据（自动匹配路径）
adata = sc.read(os.path.join(DATA_RAW, "human_core_genes.h5ad"))


## 人
### GSE104589


In [ ]:

# ==============================================
# GSE104589 表达矩阵处理 【最终完美版】
# 自动跳过样本元数据文件 | 仅处理基因表达矩阵 | 适配所有txt/txt.gz
# 路径已固定 | 直接运行
# ==============================================
import pandas as pd
import numpy as np
from mygene import MyGeneInfo
import os
import glob

# ===================== 你的固定路径（无需修改）=====================
RAW_DATA_DIR = r"./17 骨关节炎和囊泡/GEO 松动组织 vs. 稳定假体组织的基因表达谱/标本为人/GSE 104589/GSE104589_RAW"
SAVE_DIR = r"./data"
# ==================================================================

os.makedirs(SAVE_DIR, exist_ok=True)
mg = MyGeneInfo()

# 9个样本匹配（标准文件名：GSMxxx_Sample_x.txt.gz 或 .txt）
SAMPLE_MAP = {
    "GSM2803868": "GSM2803868_Sample_1",
    "GSM2803869": "GSM2803869_Sample_2",
    "GSM2803870": "GSM2803870_Sample_3",
    "GSM2803871": "GSM2803871_Sample_4",
    "GSM2803872": "GSM2803872_Sample_5",
    "GSM2803873": "GSM2803873_Sample_6",
    "GSM3670779": "GSM3670779_Sample_7",
    "GSM3670780": "GSM3670780_Sample_8",
    "GSM3670781": "GSM3670781_Sample_9",
}

print("开始处理文件，自动跳过样本元数据，仅提取基因表达矩阵...")
expr_df = None

# ---------------------- 全自动读取：跳过错误文件，只留表达矩阵 ----------------------
for gsm, file_prefix in SAMPLE_MAP.items():
    # 兼容 .txt.gz 和 .txt
    pattern = os.path.join(RAW_DATA_DIR, f"{file_prefix}.txt*")
    matches = glob.glob(pattern)
    if not matches:
        print(f"⚠️ 未找到文件：{file_prefix}.txt*，跳过")
        continue
    file_path = matches[0]
    print(f"读取文件：{os.path.basename(file_path)}")

    # 根据扩展名决定压缩方式
    compression = 'gzip' if file_path.endswith('.gz') else None
    df = pd.read_csv(
        file_path,
        sep="\t",
        compression=compression,
        header=0,
        encoding="latin-1",
        encoding_errors="replace",
        low_memory=False
    )
    df.columns = df.columns.str.strip()

    # ✅ 自动跳过【样本元数据文件】（列名含Sampletype/Macrophage）
    if "Sampletype" in df.columns or "Macrophage" in df.columns:
        print(f"⏭️  跳过样本元数据文件：{file_prefix}")
        continue

    # ✅ 仅处理【基因表达文件】（含tracking_id/FPKM）
    if "tracking_id" in df.columns and "FPKM" in df.columns:
        df = df.set_index("tracking_id")
        col = df["FPKM"].rename(gsm)

        # 合并样本
        if expr_df is None:
            expr_df = col.to_frame()
        else:
            expr_df = expr_df.join(col, how="inner")
        print(f"✅ 成功提取表达矩阵：{gsm}")
    else:
        print(f"⚠️ 文件 {file_prefix} 缺少 tracking_id 或 FPKM 列，跳过")

# ---------------------- 数据清洗与过滤 ----------------------
expr_df = expr_df.astype(np.float64).fillna(0)
# 过滤低表达基因（50%以上样本有表达）
threshold = expr_df.shape[1] * 0.5
df_filter = expr_df.loc[(expr_df > 0).sum(axis=1) >= threshold]

# ---------------------- 基因ID转换 ----------------------
try:
    res = mg.querymany(df_filter.index.tolist(), scopes=["refseq"], species="human", fields="symbol")
    id_map = {i["query"]: i.get("symbol", np.nan) for i in res}
    df_filter["symbol"] = df_filter.index.map(id_map)
    df_final = df_filter.dropna(subset=["symbol"]).groupby("symbol").mean()
except Exception as e:
    df_final = df_filter
    print(f"ℹ️  ID转换失败（网络/ID问题），保留原始基因ID。错误信息：{e}")

# ---------------------- 保存最终结果 ----------------------
# 1. 表达矩阵
matrix_path = os.path.join(SAVE_DIR, "GSE104589_原始表达矩阵.csv")
df_final.to_csv(matrix_path, index=True, encoding="utf-8-sig")

# 2. 样本分组文件
group_map = {
    "GSM2803868": "Control",
    "GSM2803869": "Control",
    "GSM2803870": "Control",
    "GSM2803871": "UHMWPE",
    "GSM2803872": "UHMWPE",
    "GSM2803873": "UHMWPE",
    "GSM3670779": "VE-UHMWPE",
    "GSM3670780": "VE-UHMWPE",
    "GSM3670781": "VE-UHMWPE"
}
group_df = pd.DataFrame({
    "sample_id": df_final.columns,
    "group": [group_map[i] for i in df_final.columns],
    "group_num": [0 if g == "Control" else 1 if g == "UHMWPE" else 2 for g in [group_map[i] for i in df_final.columns]]
})
group_path = os.path.join(SAVE_DIR, "GSE104589_样本分组信息.csv")
group_df.to_csv(group_path, index=False, encoding="utf-8-sig")

# ---------------------- 完成 ----------------------
print("\n🎉🎉🎉 全部处理完成！")
print(f"📦 表达矩阵形状：{df_final.shape}")
print(f"📂 结果保存路径：{SAVE_DIR}")
print("✅ 输出文件：GSE104589_原始表达矩阵.csv + 样本分组信息.csv")
print("⚠️  矩阵为原始FPKM，直接用于差异分析，禁止标准化！")



In [ ]:


import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

# ---------------------- rpy2 相关导入 ----------------------
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri
from rpy2.robjects.conversion import localconverter
from rpy2.robjects.packages import importr, isinstalled

# 确保 DESeq2 已安装
utils = importr('utils')
if not isinstalled('DESeq2'):
    print("正在安装 DESeq2 ...")
    utils.install_packages('BiocManager')
    ro.r('BiocManager::install("DESeq2", update=FALSE, ask=FALSE)')
deseq2 = importr('DESeq2')

# ---------------------- 差异分析函数 ----------------------
def diff_analysis(
    expr_csv: str,
    group_csv: str,
    control_group: str,
    treat_group: str,
    min_exp: float = 1,
    gse_id: str = "GSE104589"
):
    """
    使用 DESeq2 进行差异表达分析。
    输出文件命名规则：{gse_id}_差异分析_{control}_vs_{treat}.csv 等
    """
    # ===================== 1. 读取数据 =====================
    expr = pd.read_csv(expr_csv, index_col=0).copy()
    group = pd.read_csv(group_csv).copy()

    # 筛选目标分组
    target_samples = group[group["group"].isin([control_group, treat_group])]["sample_id"].tolist()
    expr = expr[target_samples].copy()
    group = group[group["sample_id"].isin(target_samples)].copy()

    control_samples = group[group["group"] == control_group]["sample_id"].tolist()
    treat_samples = group[group["group"] == treat_group]["sample_id"].tolist()

    print(f"✅ 对照组：{control_group} ({len(control_samples)}个样本)")
    print(f"✅ 处理组：{treat_group} ({len(treat_samples)}个样本)")

    # ===================== 2. 低表达过滤 =====================
    expr = expr[expr.mean(axis=1) >= min_exp].copy()
    print(f"📊 过滤后基因数：{expr.shape[0]}")

    # ===================== 3. 准备 DESeq2 输入数据 =====================
    col_data = pd.DataFrame({
        "sample": expr.columns,
        "condition": [control_group if s in control_samples else treat_group for s in expr.columns]
    })
    col_data["condition"] = col_data["condition"].astype("category")
    col_data = col_data.set_index("sample")

    design = ro.Formula('~ condition')

    # ===================== 4. 运行 DESeq2 =====================
    print("⏳ 正在运行 DESeq2 分析...")
    
    # 上下文管理器：自动完成 Pandas ↔ R 转换
    with localconverter(ro.default_converter + pandas2ri.converter):
        # 构建 DESeq2 对象
        dds = deseq2.DESeqDataSetFromMatrix(
            countData=expr.astype(int),
            colData=col_data,
            design=design
        )
        dds = deseq2.DESeq(dds)

        # 提取差异结果
        res = deseq2.results(dds, contrast=ro.StrVector(['condition', treat_group, control_group]))
        
        # 将 R 的 S4 对象转换为 data.frame，由于转换器已激活，结果直接是 Pandas DataFrame
        res_df = ro.r['as.data.frame'](res)

    # 此时 res_df 已经是 Pandas DataFrame，重置索引获取基因ID列
    res_df = res_df.reset_index().rename(columns={"index": "gene_id"})

    # ===================== 5. 整理结果 =====================
    expr["control_mean"] = expr[control_samples].mean(axis=1)
    expr["treat_mean"] = expr[treat_samples].mean(axis=1)

    result = pd.DataFrame({
        "control_mean": expr.loc[res_df["gene_id"], "control_mean"].values,
        "treat_mean": expr.loc[res_df["gene_id"], "treat_mean"].values,
        "log2FoldChange": res_df["log2FoldChange"].values,
        "P_value": res_df["pvalue"].values,
        "FDR": res_df["padj"].values
    }, index=res_df["gene_id"])

    result["FDR"] = result["FDR"].fillna(1.0)
    result["P_value"] = result["P_value"].fillna(1.0)

    result["regulate"] = np.where(
        (result["log2FoldChange"] > 1) & (result["FDR"] < 0.05), "Up",
        np.where((result["log2FoldChange"] < -1) & (result["FDR"] < 0.05), "Down", "No")
    )

    # ===================== 6. 保存文件 =====================
    output_dir = "./data/"
    
    result_path = f"{output_dir}{gse_id}_差异分析_{control_group}_vs_{treat_group}.csv"
    result.to_csv(result_path, encoding="utf-8-sig")
    
    up_genes = result[result["regulate"] == "Up"].copy()
    up_path = f"{output_dir}{gse_id}_上调基因_{control_group}_vs_{treat_group}.csv"
    up_genes.to_csv(up_path, encoding="utf-8-sig")
    
    down_genes = result[result["regulate"] == "Down"].copy()
    down_path = f"{output_dir}{gse_id}_下调基因_{control_group}_vs_{treat_group}.csv"
    down_genes.to_csv(down_path, encoding="utf-8-sig")

    print(f"\n🎉 {gse_id} 差异分析完成！")
    print(f"📈 上调基因：{len(up_genes)} 个")
    print(f"📉 下调基因：{len(down_genes)} 个")
    print(f"📂 文件保存至：{output_dir}")
    print("-" * 70)

    return result


# ==============================================================================
# 【一键运行】
# ==============================================================================
if __name__ == "__main__":
    GSE_ID = "GSE104589"
    EXPR_PATH = f"./data/{GSE_ID}_原始表达矩阵.csv"
    GROUP_PATH = f"./data/{GSE_ID}_样本分组信息.csv"
    OUTPUT_DIR = "./data/"

    # 分析 Control vs UHMWPE
    diff_analysis(
        expr_csv=EXPR_PATH,
        group_csv=GROUP_PATH,
        control_group="Control",
        treat_group="UHMWPE",
        gse_id=GSE_ID
    )

    # 分析 Control vs VE-UHMWPE
    diff_analysis(
        expr_csv=EXPR_PATH,
        group_csv=GROUP_PATH,
        control_group="Control",
        treat_group="VE-UHMWPE",
        gse_id=GSE_ID
    )

    # 合并结果（确保文件存在后再读取，可加 try-except）
    diff_uhmwpe = pd.read_csv(f"{OUTPUT_DIR}{GSE_ID}_差异分析_Control_vs_UHMWPE.csv", encoding="utf-8-sig")
    diff_ve = pd.read_csv(f"{OUTPUT_DIR}{GSE_ID}_差异分析_Control_vs_VE-UHMWPE.csv", encoding="utf-8-sig")
    diff_uhmwpe["comparison_group"] = "Control_vs_UHMWPE"
    diff_ve["comparison_group"] = "Control_vs_VE-UHMWPE"
    diff_all = pd.concat([diff_uhmwpe, diff_ve], ignore_index=True)
    diff_all.to_csv(f"{OUTPUT_DIR}{GSE_ID}_差异分析.csv", index=False, encoding="utf-8-sig")

    up_uhmwpe = pd.read_csv(f"{OUTPUT_DIR}{GSE_ID}_上调基因_Control_vs_UHMWPE.csv", encoding="utf-8-sig")
    up_ve = pd.read_csv(f"{OUTPUT_DIR}{GSE_ID}_上调基因_Control_vs_VE-UHMWPE.csv", encoding="utf-8-sig")
    up_uhmwpe["comparison_group"] = "Control_vs_UHMWPE"
    up_ve["comparison_group"] = "Control_vs_VE-UHMWPE"
    up_all = pd.concat([up_uhmwpe, up_ve], ignore_index=True)
    up_all.to_csv(f"{OUTPUT_DIR}{GSE_ID}_上调基因.csv", index=False, encoding="utf-8-sig")

    down_uhmwpe = pd.read_csv(f"{OUTPUT_DIR}{GSE_ID}_下调基因_Control_vs_UHMWPE.csv", encoding="utf-8-sig")
    down_ve = pd.read_csv(f"{OUTPUT_DIR}{GSE_ID}_下调基因_Control_vs_VE-UHMWPE.csv", encoding="utf-8-sig")
    down_uhmwpe["comparison_group"] = "Control_vs_UHMWPE"
    down_ve["comparison_group"] = "Control_vs_VE-UHMWPE"
    down_all = pd.concat([down_uhmwpe, down_ve], ignore_index=True)
    down_all.to_csv(f"{OUTPUT_DIR}{GSE_ID}_下调基因.csv", index=False, encoding="utf-8-sig")

    print("\n" + "="*70)
    print(f"🎉 所有文件均带 {GSE_ID} 前缀，合并完成！")
    print(f"📊 总差异分析文件：{GSE_ID}_差异分析.csv")
    print(f"📈 总上调基因文件：{GSE_ID}_上调基因.csv")
    print(f"📉 总下调基因文件：{GSE_ID}_下调基因.csv")
    print(f"📂 所有文件保存至：{OUTPUT_DIR}")
    print("="*70)




In [ ]:
# -*- coding: utf-8 -*-
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
plt.rcParams['font.sans-serif'] = ['DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# ===================== 基础配置 =====================
OUTPUT_DIR = "./data/"
GSE_ID = "GSE104589"
FC_THRESH = 1
FDR_THRESH = 0.05

# ===================== 函数：绘制火山图（自适应列名） =====================
def plot_volcano(input_file, save_name, title_name):
    # 1. 读取差异分析数据
    df = pd.read_csv(input_file, index_col=0)
    
    # 2. 自动识别 log2FC 列名（兼容 log2FC 和 log2FoldChange）
    if 'log2FC' in df.columns:
        log2fc_col = 'log2FC'
    elif 'log2FoldChange' in df.columns:
        log2fc_col = 'log2FoldChange'
    else:
        raise KeyError(f"文件中未找到 log2FC 或 log2FoldChange 列，现有列：{df.columns.tolist()}")
    
    # 3. 自动识别 FDR 列名（兼容 FDR 和 padj）
    if 'FDR' in df.columns:
        fdr_col = 'FDR'
    elif 'padj' in df.columns:
        fdr_col = 'padj'
    else:
        raise KeyError(f"文件中未找到 FDR 或 padj 列，现有列：{df.columns.tolist()}")
    
    # 4. 统一重命名为标准列名（方便后续代码）
    df.rename(columns={log2fc_col: 'log2FC', fdr_col: 'FDR'}, inplace=True)
    
    # 5. 计算 Y 轴：-log10(FDR)，并处理 FDR 为 0 的情况（极小值替换）
    df['FDR'] = df['FDR'].astype(float)
    df['FDR'] = df['FDR'].replace(0, 1e-300)  # 避免 -log10(0) 无穷大
    df['-log10(FDR)'] = -np.log10(df['FDR'])
    
    # 6. 基因分类（Up/Down/No）
    def classify(row):
        if row['FDR'] < FDR_THRESH and row['log2FC'] > FC_THRESH:
            return 'Up'
        elif row['FDR'] < FDR_THRESH and row['log2FC'] < -FC_THRESH:
            return 'Down'
        else:
            return 'No'
    df['type'] = df.apply(classify, axis=1)
    
    # 7. 绘图
    plt.figure(figsize=(6, 5), dpi=300)
    colors = {'Up': '#FF4C4C', 'Down': '#4C6EFF', 'No': '#B8B8B8'}
    sns.scatterplot(x='log2FC', y='-log10(FDR)', hue='type', 
                    data=df, palette=colors, s=8, alpha=0.8, legend='full')
    
    # 添加阈值线
    plt.axvline(x=FC_THRESH, color='gray', linestyle='--', lw=0.8)
    plt.axvline(x=-FC_THRESH, color='gray', linestyle='--', lw=0.8)
    plt.axhline(y=-np.log10(FDR_THRESH), color='gray', linestyle='--', lw=0.8)
    
    # 图表设置
    plt.title(f'{GSE_ID} {title_name}', fontsize=12, pad=15)
    plt.xlabel('log2(Fold Change)', fontsize=10)
    plt.ylabel('-log10(FDR)', fontsize=10)
    
    # 根据数据范围动态调整 x 轴范围（可选，保留 -6~6 或自动）
    max_fc = df['log2FC'].abs().max()
    xlim = max(6, max_fc + 0.5) if max_fc > 6 else 6
    plt.xlim(-xlim, xlim)
    
    plt.legend(title='Gene Type', loc='upper right')
    plt.tight_layout()
    
    # 保存高清图片
    plt.savefig(f"{OUTPUT_DIR}{save_name}.png", dpi=300, bbox_inches='tight')
    plt.close()
    
    # 统计输出
    up_num = len(df[df['type']=='Up'])
    down_num = len(df[df['type']=='Down'])
    print(f"✅ {title_name} 火山图完成 | 上调:{up_num} | 下调:{down_num}")

# ===================== 一键绘制两组火山图 =====================
if __name__ == "__main__":
    print("🚀 开始绘制 GSE104589 火山图...\n")
    
    # 1. Control vs UHMWPE 火山图
    plot_volcano(
        input_file=f"{OUTPUT_DIR}{GSE_ID}_差异分析_Control_vs_UHMWPE.csv",
        save_name=f"{GSE_ID}_火山图_Control_vs_UHMWPE",
        title_name="Volcano Plot (Control vs UHMWPE)"
    )
    
    # 2. Control vs VE-UHMWPE 火山图
    plot_volcano(
        input_file=f"{OUTPUT_DIR}{GSE_ID}_差异分析_Control_vs_VE-UHMWPE.csv",
        save_name=f"{GSE_ID}_火山图_Control_vs_VE-UHMWPE",
        title_name="Volcano Plot (Control vs VE-UHMWPE)"
    )
    
    print(f"\n🎉 所有火山图已保存至：{OUTPUT_DIR}")
# -*- coding: utf-8 -*-
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Patch

# ===================== 基础配置 =====================
OUTPUT_DIR = "./data/"
GSE_ID = "GSE104589"
FC_THRESH = 1
FDR_THRESH = 0.05

# SCI标准热图配色
cmap = LinearSegmentedColormap.from_list('custom', ['#3A77AF', 'white', '#FF3D3D'])
plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 300

def plot_heatmap(comparison):
    """
    comparison: 对比组名称，例如 "UHMWPE" 或 "VE-UHMWPE"
    对应的差异文件: {GSE_ID}_差异分析_Control_vs_{comparison}.csv
    """
    # 1. 读取核心数据
    expr = pd.read_csv(f"{OUTPUT_DIR}{GSE_ID}_原始表达矩阵.csv", index_col=0)
    group = pd.read_csv(f"{OUTPUT_DIR}{GSE_ID}_样本分组信息.csv")
    deg = pd.read_csv(f"{OUTPUT_DIR}{GSE_ID}_差异分析_Control_vs_{comparison}.csv", index_col=0)
    
    # 2. 自动识别列名（兼容 log2FC/log2FoldChange 和 FDR/padj）
    if 'log2FC' in deg.columns:
        log2fc_col = 'log2FC'
    elif 'log2FoldChange' in deg.columns:
        log2fc_col = 'log2FoldChange'
    else:
        raise KeyError(f"未找到 log2FC 或 log2FoldChange，现有列：{deg.columns.tolist()}")
    
    if 'FDR' in deg.columns:
        fdr_col = 'FDR'
    elif 'padj' in deg.columns:
        fdr_col = 'padj'
    else:
        raise KeyError(f"未找到 FDR 或 padj，现有列：{deg.columns.tolist()}")
    
    deg.rename(columns={log2fc_col: 'log2FC', fdr_col: 'FDR'}, inplace=True)
    
    # 3. 筛选显著差异基因
    sig_genes = deg[
        (deg['FDR'] < FDR_THRESH) & 
        (abs(deg['log2FC']) > FC_THRESH)
    ].index.tolist()
    
    if len(sig_genes) == 0:
        print(f"❌ {comparison} 无显著差异基因，无法绘制热图")
        return
    print(f"✅ {comparison} 提取到 {len(sig_genes)} 个显著差异基因")
    
    # 4. 提取表达矩阵 + 行Z-score标准化
    heatmap_data = expr.loc[sig_genes, :].copy()
    heatmap_data = heatmap_data.apply(lambda x: (x - x.mean()) / x.std(), axis=1)
    
    # 5. 样本分组注释
    sample_order = heatmap_data.columns.tolist()
    group_dict = dict(zip(group['sample_id'], group['group']))
    group_colors = {
        'Control': '#8DD3C7',
        'UHMWPE': '#FFB347',
        'VE-UHMWPE': '#BEBADA'
    }
    col_colors = [group_colors[group_dict[s]] for s in sample_order]
    
    # 6. 绘制聚类热图
    g = sns.clustermap(
        heatmap_data,
        cmap=cmap,
        row_cluster=True,
        col_cluster=True,
        col_colors=col_colors,
        linewidths=0.05,
        figsize=(11, 8),
        dendrogram_ratio=(0.12, 0.1),
        cbar_kws={"label": "Z-score", "orientation": "vertical"}
    )
    
    g.fig.suptitle(f'{GSE_ID} Differential Gene Expression Heatmap (Control vs {comparison})', y=1.02, fontsize=14)
    g.ax_heatmap.set_xlabel('Samples', fontsize=10, labelpad=10)
    g.ax_heatmap.set_ylabel('Differential Genes', fontsize=10, labelpad=10)
    
    legend_elements = [
        Patch(facecolor=group_colors['Control'], label='Control'),
        Patch(facecolor=group_colors['UHMWPE'], label='UHMWPE'),
        Patch(facecolor=group_colors['VE-UHMWPE'], label='VE-UHMWPE')
    ]
    g.ax_col_dendrogram.legend(handles=legend_elements, loc='upper right', bbox_to_anchor=(1.2, 1))
    
    # 7. 保存图片（文件名包含对比组）
    save_name = f"{OUTPUT_DIR}{GSE_ID}_差异基因热图_Control_vs_{comparison}.png"
    plt.savefig(save_name, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"🎉 热图已保存：{save_name}")

if __name__ == "__main__":
    print("🚀 开始绘制基于 DESeq2 的差异基因热图...\n")
    # 绘制两组对比的热图
    plot_heatmap("UHMWPE")
    plot_heatmap("VE-UHMWPE")
    print("\n🎉 所有热图绘制完成！")
# -*- coding: utf-8 -*-
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Patch

# ===================== 基础配置 =====================
OUTPUT_DIR = "./data/"
GSE_ID = "GSE104589"
FC_THRESH = 1
FDR_THRESH = 0.05

# SCI标准热图配色
cmap = LinearSegmentedColormap.from_list('custom', ['#3A77AF', 'white', '#FF3D3D'])
plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 300

def plot_heatmap(comparison):
    """
    comparison: 对比组名称，例如 "UHMWPE" 或 "VE-UHMWPE"
    对应的差异文件: {GSE_ID}_差异分析_Control_vs_{comparison}.csv
    """
    # 1. 读取核心数据
    expr = pd.read_csv(f"{OUTPUT_DIR}{GSE_ID}_原始表达矩阵.csv", index_col=0)
    group = pd.read_csv(f"{OUTPUT_DIR}{GSE_ID}_样本分组信息.csv")
    deg = pd.read_csv(f"{OUTPUT_DIR}{GSE_ID}_差异分析_Control_vs_{comparison}.csv", index_col=0)
    
    # 2. 自动识别列名（兼容 log2FC/log2FoldChange 和 FDR/padj）
    if 'log2FC' in deg.columns:
        log2fc_col = 'log2FC'
    elif 'log2FoldChange' in deg.columns:
        log2fc_col = 'log2FoldChange'
    else:
        raise KeyError(f"未找到 log2FC 或 log2FoldChange，现有列：{deg.columns.tolist()}")
    
    if 'FDR' in deg.columns:
        fdr_col = 'FDR'
    elif 'padj' in deg.columns:
        fdr_col = 'padj'
    else:
        raise KeyError(f"未找到 FDR 或 padj，现有列：{deg.columns.tolist()}")
    
    deg.rename(columns={log2fc_col: 'log2FC', fdr_col: 'FDR'}, inplace=True)
    
    # 3. 筛选显著差异基因
    sig_genes = deg[
        (deg['FDR'] < FDR_THRESH) & 
        (abs(deg['log2FC']) > FC_THRESH)
    ].index.tolist()
    
    if len(sig_genes) == 0:
        print(f"❌ {comparison} 无显著差异基因，无法绘制热图")
        return
    print(f"✅ {comparison} 提取到 {len(sig_genes)} 个显著差异基因")
    
    # 4. 提取表达矩阵 + 行Z-score标准化
    heatmap_data = expr.loc[sig_genes, :].copy()
    heatmap_data = heatmap_data.apply(lambda x: (x - x.mean()) / x.std(), axis=1)
    
    # 5. 样本分组注释
    sample_order = heatmap_data.columns.tolist()
    group_dict = dict(zip(group['sample_id'], group['group']))
    group_colors = {
        'Control': '#8DD3C7',
        'UHMWPE': '#FFB347',
        'VE-UHMWPE': '#BEBADA'
    }
    col_colors = [group_colors[group_dict[s]] for s in sample_order]
    
    # 6. 绘制聚类热图
    g = sns.clustermap(
        heatmap_data,
        cmap=cmap,
        row_cluster=True,
        col_cluster=True,
        col_colors=col_colors,
        linewidths=0.05,
        figsize=(11, 8),
        dendrogram_ratio=(0.12, 0.1),
        cbar_kws={"label": "Z-score", "orientation": "vertical"}
    )
    
    g.fig.suptitle(f'{GSE_ID} Differential Gene Expression Heatmap (Control vs {comparison})', y=1.02, fontsize=14)
    g.ax_heatmap.set_xlabel('Samples', fontsize=10, labelpad=10)
    g.ax_heatmap.set_ylabel('Differential Genes', fontsize=10, labelpad=10)
    
    legend_elements = [
        Patch(facecolor=group_colors['Control'], label='Control'),
        Patch(facecolor=group_colors['UHMWPE'], label='UHMWPE'),
        Patch(facecolor=group_colors['VE-UHMWPE'], label='VE-UHMWPE')
    ]
    g.ax_col_dendrogram.legend(handles=legend_elements, loc='upper right', bbox_to_anchor=(1.2, 1))
    
    # 7. 保存图片（文件名包含对比组）
    save_name = f"{OUTPUT_DIR}{GSE_ID}_差异基因热图_Control_vs_{comparison}.png"
    plt.savefig(save_name, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"🎉 热图已保存：{save_name}")

if __name__ == "__main__":
    print("🚀 开始绘制基于 DESeq2 的差异基因热图...\n")
    # 绘制两组对比的热图
    plot_heatmap("UHMWPE")
    plot_heatmap("VE-UHMWPE")
    print("\n🎉 所有热图绘制完成！")




### GSE149315 

In [ ]:


# ==============================================
# GSE149315 circRNA表达矩阵处理（适配差异分析/可视化）
# 使用 limma 标准流程（适用于芯片数据）
# 输出原始未标准化矩阵 | 支持Excel输入
# 路径已固定 | 直接运行 | 新增上调/下调circRNA输出
# 阈值：标准严格阈值 |log2FC|>1 + FDR<0.05
# ==============================================
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings("ignore")

# ---------------------- rpy2 相关导入 ----------------------
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri
from rpy2.robjects.conversion import localconverter
from rpy2.robjects.packages import importr, isinstalled

# 确保 limma 已安装（自动安装）
utils = importr('utils')
if not isinstalled('limma'):
    print("正在安装 limma ...")
    utils.install_packages('BiocManager')
    ro.r('BiocManager::install("limma", update=FALSE, ask=FALSE)')

# ===================== 固定路径 =====================
RAW_DATA_PATH = r"./17 骨关节炎和囊泡/GEO 松动组织 vs. 稳定假体组织的基因表达谱/标本为人/GSE 149315/GSE149315_Expression_circRNA.xlsx"
SAVE_DIR = r"./data"
# ==================================================================

os.makedirs(SAVE_DIR, exist_ok=True)

# ---------------------- 1. 读取并整理circRNA表达数据 ----------------------
print("开始读取circRNA表达数据...")
expr_df = pd.read_excel(RAW_DATA_PATH, index_col="circRNA_ID")

sample_cols = expr_df.columns.tolist()
print(f"检测到样本：{sample_cols}")
print(f"原始数据形状：{expr_df.shape}（circRNA数量 × 样本数量）")

# ---------------------- 2. 样本分组定义 ----------------------
group_map = {
    "T1": "Loose", "T2": "Loose", "T3": "Loose",
    "N1": "Stable", "N2": "Stable", "N3": "Stable"
}
group_num_map = {"Stable": 0, "Loose": 1}

if not all(col in group_map for col in sample_cols):
    raise ValueError("部分样本未定义分组！请检查group_map与样本列是否匹配")

# ---------------------- 3. 数据清洗与低表达过滤 ----------------------
print("开始数据清洗与低表达过滤...")
expr_df = expr_df.astype(np.float64).fillna(0)

# 芯片数据过滤：保留至少在2个样本中表达值 > 0 的circRNA
# （可根据需要调整，此处保留至少50%样本表达>0，与原逻辑一致）
sample_count = expr_df.shape[1]
threshold = sample_count * 0.5
df_filter = expr_df.loc[(expr_df > 0).sum(axis=1) >= threshold]

print(f"过滤后数据形状：{df_filter.shape}（保留的circRNA数量 × 样本数量）")
print(f"过滤掉低表达circRNA数量：{expr_df.shape[0] - df_filter.shape[0]}")

df_final = df_filter.copy()
print("保留circRNA ID作为行名，直接用于差异分析")

# 注意：芯片数据通常已经过 log2 转换和归一化，无需额外处理
# 如果您的数据是原始信号强度，建议自行进行 log2 转换

# ---------------------- 4. 准备设计矩阵 ----------------------
group_labels = [group_map[col] for col in df_final.columns]
# 设计矩阵：截距 + 分组（Loose vs Stable）
design_matrix = pd.DataFrame({
    'Intercept': 1,
    'Loose_vs_Stable': [1 if g == 'Loose' else 0 for g in group_labels]
}).astype(float)

# ---------------------- 5. 差异分析（标准 limma 流程，无 trend） ----------------------
print("\n开始差异分析（limma 标准流程，适用于芯片数据）...")

with localconverter(ro.default_converter + pandas2ri.converter):
    # 将数据传入 R 全局环境（自动转换为 R 对象）
    ro.globalenv['expr_mat'] = df_final
    ro.globalenv['design_mat'] = design_matrix
    
    # 执行 R 代码
    ro.r('''
        library(limma)
        # 线性拟合
        fit <- lmFit(expr_mat, design_mat)
        # 经验贝叶斯收缩（标准方法，不设 trend）
        fit <- eBayes(fit)
        # 提取对比组结果（coef=2 对应 Loose_vs_Stable）
        res <- topTable(fit, coef=2, number=nrow(expr_mat), sort.by="none")
    ''')
    
    # 获取结果（自动转换为 pandas DataFrame）
    res_df = ro.globalenv['res']

# 提取所需列
log2fc = res_df['logFC'].values
p_values = res_df['P.Value'].values
fdr = res_df['adj.P.Val'].values

# 整合差异分析结果
diff_result = pd.DataFrame({
    "circRNA_ID": df_final.index,
    "log2FC": log2fc,
    "p_value": p_values,
    "FDR": fdr
}).set_index("circRNA_ID")

# ---------------------- 6. 筛选显著上调/下调circRNA ----------------------
up_circ = diff_result[(diff_result["log2FC"] > 1) & (diff_result["FDR"] < 0.05)]
down_circ = diff_result[(diff_result["log2FC"] < -1) & (diff_result["FDR"] < 0.05)]

print(f"\n📊 显著差异circRNA统计（标准严格阈值 |log2FC|>1 + FDR<0.05）：")
print(f"✅ 上调circRNA数量：{len(up_circ)}")
print(f"✅ 下调circRNA数量：{len(down_circ)}")

# ---------------------- 7. 保存所有结果文件 ----------------------
print("\n保存表达矩阵、分组信息、差异结果、上调/下调circRNA...")

# 7.1 原始表达矩阵
matrix_path = os.path.join(SAVE_DIR, "GSE149315_circRNA_原始表达矩阵.csv")
df_final.to_csv(matrix_path, index=True, encoding="utf-8-sig")

# 7.2 样本分组信息
group_path = os.path.join(SAVE_DIR, "GSE149315_样本分组信息.csv")
group_df = pd.DataFrame({
    "sample_id": df_final.columns,
    "group": [group_map[col] for col in df_final.columns],
    "group_num": [group_num_map[group_map[col]] for col in df_final.columns]
})
group_df.to_csv(group_path, index=False, encoding="utf-8-sig")

# 7.3 完整差异分析结果（新增）
diff_path = os.path.join(SAVE_DIR, "GSE149315_circRNA_差异分析结果.csv")
diff_result.to_csv(diff_path, index=True, encoding="utf-8-sig")
print(f"✅ 完整差异分析结果已保存：{diff_path}")

# 7.4 显著上调circRNA
up_path = os.path.join(SAVE_DIR, "GSE149315_显著上调circRNA.csv")
up_circ.to_csv(up_path, encoding="utf-8-sig")

# 7.5 显著下调circRNA
down_path = os.path.join(SAVE_DIR, "GSE149315_显著下调circRNA.csv")
down_circ.to_csv(down_path, encoding="utf-8-sig")

print("\n🎉🎉🎉 全部处理完成（使用 limma 标准流程，适用于芯片数据）！")
print(f"📦 所有文件已保存至：{SAVE_DIR}")
print(f"📄 文件列表：")
print(f"   1. 原始表达矩阵：GSE149315_circRNA_原始表达矩阵.csv")
print(f"   2. 样本分组信息：GSE149315_样本分组信息.csv")
print(f"   3. 完整差异结果：GSE149315_circRNA_差异分析结果.csv")
print(f"   4. 显著上调circRNA：GSE149315_显著上调circRNA.csv")
print(f"   5. 显著下调circRNA：GSE149315_显著下调circRNA.csv")
print("📊 差异分析方法：limma (linear model + empirical Bayes), 适用于芯片表达数据")



In [ ]:


# ==============================================
# GSE149315 circRNA 火山图 + 热图绘制
# 基于已有的差异分析结果文件
# ==============================================
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from adjustText import adjust_text

# ===================== 路径配置（与差异分析代码保持一致） =====================
GSE_ID = "GSE149315"
SAVE_DIR = "./data"

# 定义文件路径（完全对齐差异分析输出文件名）
diff_file = os.path.join(SAVE_DIR, f"{GSE_ID}_circRNA_差异分析结果.csv")
expr_file = os.path.join(SAVE_DIR, f"{GSE_ID}_circRNA_原始表达矩阵.csv")
group_file = os.path.join(SAVE_DIR, f"{GSE_ID}_样本分组信息.csv")

# 阈值参数
FC_THRESH = 1.0
FDR_THRESH = 0.05

# ===================== 1. 读取数据 =====================
print("读取差异分析结果...")
diff_result = pd.read_csv(diff_file, index_col=0)

print("读取原始表达矩阵...")
expr_df = pd.read_csv(expr_file, index_col=0)

print("读取样本分组信息...")
group_df = pd.read_csv(group_file)

# 确保样本顺序一致
expr_df = expr_df[group_df["sample_id"]]

# 构建分组字典
group_map = dict(zip(group_df["sample_id"], group_df["group"]))
group_num_map = {"Stable": 0, "Loose": 1}  # 根据实际分组调整

# ===================== 2. 定义显著性标签 =====================
diff_result["is_significant"] = (abs(diff_result["log2FC"]) > FC_THRESH) & (diff_result["FDR"] < FDR_THRESH)
significant_circ = diff_result["is_significant"].sum()
print(f"显著差异circRNA数量：{significant_circ}")

# ===================== 3. 绘制火山图 =====================
print("\n绘制火山图...")
plt.figure(figsize=(10, 6))
plt.rcParams["font.sans-serif"] = ["Arial"]

non_sig = diff_result[~diff_result["is_significant"]]
sig_up = diff_result[(diff_result["is_significant"]) & (diff_result["log2FC"] > 0)]
sig_down = diff_result[(diff_result["is_significant"]) & (diff_result["log2FC"] < 0)]

plt.scatter(non_sig["log2FC"], -np.log10(non_sig["FDR"]), c="gray", alpha=0.5, s=20, label="NS")
plt.scatter(sig_up["log2FC"], -np.log10(sig_up["FDR"]), c="red", alpha=0.8, s=30, label=f"Up ({len(sig_up)})")
plt.scatter(sig_down["log2FC"], -np.log10(sig_down["FDR"]), c="blue", alpha=0.8, s=30, label=f"Down ({len(sig_down)})")

# 阈值线
plt.axvline(x=FC_THRESH, color="black", linestyle="--", alpha=0.5)
plt.axvline(x=-FC_THRESH, color="black", linestyle="--", alpha=0.5)
plt.axhline(y=-np.log10(FDR_THRESH), color="black", linestyle="--", alpha=0.5)

# 标注Top10显著circRNA（按FDR排序）
top10 = diff_result.sort_values("FDR").head(10)
texts = []
for circ_id in top10.index:
    x = top10.loc[circ_id, "log2FC"]
    y = -np.log10(top10.loc[circ_id, "FDR"])
    # 截断过长的circRNA ID
    label = circ_id if len(circ_id) <= 30 else circ_id[:27] + "..."
    texts.append(plt.text(x, y, label, fontsize=8))
adjust_text(texts, arrowprops=dict(arrowstyle="->", color="black", alpha=0.3))

plt.xlabel("log2(Fold Change)", fontsize=12)
plt.ylabel("-log10(FDR)", fontsize=12)
plt.title(f"{GSE_ID} circRNA Volcano Plot (Loose vs Stable)", fontsize=14, fontweight="bold")
plt.legend()
plt.grid(alpha=0.3)

volcano_path = os.path.join(SAVE_DIR, f"{GSE_ID}_circRNA_火山图.png")
plt.tight_layout()
plt.savefig(volcano_path, dpi=300, bbox_inches="tight")
plt.close()
print(f"火山图已保存：{volcano_path}")

# ===================== 4. 绘制热图（Top50差异circRNA） =====================
if significant_circ >= 1:
    print("\n绘制热图...")
    # 取Top50差异circRNA（按FDR排序）
    top_n = min(50, diff_result.shape[0])
    top_circ = diff_result.sort_values("FDR").head(top_n).index
    heat_data = expr_df.loc[top_circ].copy()
    
    # Z-score 标准化（按行）
    heat_scaled = (heat_data - heat_data.mean(axis=1).values.reshape(-1,1)) / heat_data.std(axis=1).values.reshape(-1,1)
    
    # 样本分组颜色条
    sample_colors = [plt.cm.Set3(group_num_map[group_map[col]]) for col in heat_data.columns]
    
    # 绘制聚类热图（仅行聚类，不列聚类）
    g = sns.clustermap(heat_scaled, cmap="RdBu_r", row_cluster=True, col_cluster=False,
                       col_colors=sample_colors,
                       cbar_kws={"label": "Z-score"}, linewidths=0.1, figsize=(12, 10))
    g.ax_col_dendrogram.set_visible(False)
    g.fig.suptitle(f"Top {top_n} Differentially Expressed circRNA", y=1.02)
    
    heatmap_path = os.path.join(SAVE_DIR, f"{GSE_ID}_circRNA_热图.png")
    plt.savefig(heatmap_path, dpi=300, bbox_inches="tight")
    plt.close()
    print(f"热图已保存：{heatmap_path}")
else:
    print("无显著差异circRNA，跳过热图绘制")
print("\n🎉 可视化完成！")



In [ ]:
import pandas as pd
import warnings

warnings.filterwarnings('ignore')

# ===================== 配置文件路径 =====================
UP_FILE = "./data/GSE149315_显著上调circRNA.csv"
DOWN_FILE = "./data/GSE149315_显著下调circRNA.csv"
OUT_UP = "./data/GSE149315_显著上调circRNA_GeneSymbol.csv"
OUT_DOWN = "./data/GSE149315_显著下调circRNA_GeneSymbol.csv"

# 本地 circBase 映射文件
MAP_FILE = "./17 骨关节炎和囊泡/GEO 松动组织 vs. 稳定假体组织的基因表达谱/标本为人/GSE 149315/hsa_hg19_circRNA.txt"

# ===================== 加载本地映射表 =====================
print("📥 正在加载本地 circRNA 映射表...")
try:
    # 根据文件内容手动指定列名（第一行为注释行，跳过）
    col_names = [
        'chrom', 'start', 'end', 'strand', 'circRNA ID',
        'genomic length', 'spliced seq length', 'samples', 'repeats',
        'annotation', 'best transcript', 'gene symbol', 'circRNA study'
    ]
    # 跳过以 '#' 开头的注释行，使用自定义列名，无表头行
    mapping_df = pd.read_csv(MAP_FILE, sep="\t", comment='#', names=col_names, header=None)
    # 构建映射字典
    circ_to_gene = dict(zip(mapping_df['circRNA ID'], mapping_df['gene symbol']))
    print(f"✅ 成功加载 {len(circ_to_gene)} 条映射记录")
except Exception as e:
    print(f"❌ 加载映射表失败: {e}")
    # 如果仍然失败，打印文件前几行用于调试
    with open(MAP_FILE, 'r') as f:
        print("文件前5行:")
        for i, line in enumerate(f):
            if i < 5:
                print(repr(line))
            else:
                break
    raise SystemExit(1)

# ===================== 批量处理函数 =====================
def process_file(input_path, output_path):
    df = pd.read_csv(input_path)
    print(f"📂 读取文件：{input_path}")
    print(f"🔢 待转换 circRNA 数量：{len(df)}")

    # 确定 circRNA ID 列名（原文件通常为 'circRNA_ID'）
    circ_col = "circRNA_ID" if "circRNA_ID" in df.columns else df.columns[0]
    circ_ids = df[circ_col].tolist()

    genes = []
    for idx, circ_id in enumerate(circ_ids, 1):
        gene = circ_to_gene.get(circ_id, "Not_Found")
        print(f"  {idx}/{len(circ_ids)} | {circ_id} → {gene}")
        genes.append(gene)

    df['gene_symbol'] = genes
    df.to_csv(output_path, index=False, encoding='utf-8')
    print(f"✅ 转换完成！文件保存至：{output_path}\n")

# ===================== 主程序 =====================
if __name__ == "__main__":
    print("="*60)
    print("🚀 circRNA ID → 宿主基因 转换工具 (基于本地 circBase 文件)")
    print("="*60)
    process_file(UP_FILE, OUT_UP)
    process_file(DOWN_FILE, OUT_DOWN)
    print("🎉 全部转换完成！")

### GSE171542

In [ ]:
import pandas as pd
import numpy as np
from mygene import MyGeneInfo
import os
import warnings
warnings.filterwarnings('ignore')

# ---------------------- rpy2 相关导入 ----------------------
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri
from rpy2.robjects.conversion import localconverter
from rpy2.robjects.packages import importr, isinstalled

# 确保 limma 已安装
utils = importr('utils')
if not isinstalled('limma'):
    print("正在安装 limma ...")
    utils.install_packages('BiocManager')
    ro.r('BiocManager::install("limma", update=FALSE, ask=FALSE)')

# ===================== 固定路径 =====================
RAW_DATA_DIR = r"./17 骨关节炎和囊泡/GEO 松动组织 vs. 稳定假体组织的基因表达谱/标本为人/GSE 171542/GSE171542_RAW"
SAVE_DIR = r"./data"
os.makedirs(SAVE_DIR, exist_ok=True)

mg = MyGeneInfo()

# ---------------------- 样本映射与分组（明确三组）--------------------
# 定义每个样本的原始分组（Control, RANKL, TYMP）
SAMPLE_GROUP = {
    "GSM5226831": "Control",   # MCSF1
    "GSM5226832": "Control",   # MCSF2
    "GSM5226833": "Control",   # MCSF4
    "GSM5226834": "RANKL",     # RANKL1
    "GSM5226835": "RANKL",     # RANKL2
    "GSM5226836": "RANKL",     # RANKL3
    "GSM5226837": "TYMP",      # TP1
    "GSM5226838": "TYMP",      # TP2
    "GSM5226839": "TYMP",      # TP3
}

MIN_EXP_SAMPLES = 2
P_THRESHOLD = 0.05
LOG2FC_THRESHOLD = 1.0

# ---------------------- 1. 读取数据构建 FPKM 矩阵 ----------------------
print("="*50)
print("读取原始 isoforms 文件...")
expr_df = None
for gsm, group in SAMPLE_GROUP.items():
    # 根据 gsm 获取文件名（需要从 SAMPLE_GROUP 中查找，但文件名映射需单独定义）
    # 原 SAMPLE_MAP 需保留，但可以直接用文件名模板构建
    filename = f"{gsm}_{group}_trim.isoforms.results.txt.gz"  # 实际文件名可能不是这样，需用原映射
    # 为避免错误，沿用原 SAMPLE_MAP 字典，但需要提前定义好映射关系
    # 这里直接使用原 SAMPLE_MAP 中的文件名（因为原代码已经正确对应）
    # 我们重新构造一个简单的文件名映射，基于已知模式：
    file_map = {
        "GSM5226831": "GSM5226831_MCSF1_trim.isoforms.results.txt.gz",
        "GSM5226832": "GSM5226832_MCSF2_trim.isoforms.results.txt.gz",
        "GSM5226833": "GSM5226833_MCSF4_trim.isoforms.results.txt.gz",
        "GSM5226834": "GSM5226834_RANKL1_trim.isoforms.results.txt.gz",
        "GSM5226835": "GSM5226835_RANKL2_trim.isoforms.results.txt.gz",
        "GSM5226836": "GSM5226836_RANKL3_trim.isoforms.results.txt.gz",
        "GSM5226837": "GSM5226837_TP1_trim.isoforms.results.txt.gz",
        "GSM5226838": "GSM5226838_TP2_trim.isoforms.results.txt.gz",
        "GSM5226839": "GSM5226839_TP3_trim.isoforms.results.txt.gz",
    }
    file_path = os.path.join(RAW_DATA_DIR, file_map[gsm])
    df = pd.read_csv(file_path, sep="\t", compression="gzip", low_memory=False)
    gene_level = df.groupby("gene_id")["FPKM"].sum().rename(gsm)
    expr_df = gene_level.to_frame() if expr_df is None else expr_df.join(gene_level, how="inner")

print(f"原始矩阵形状：{expr_df.shape}")

# ---------------------- 2. 过滤低表达基因 ----------------------
expr_filter = expr_df.loc[(expr_df > 0).sum(axis=1) >= MIN_EXP_SAMPLES]
expr_filter = expr_filter.fillna(0).astype(float)
print(f"过滤后形状：{expr_filter.shape}")

# ---------------------- 3. 基因 ID 转换（可选）---------------------
print("转换基因 ID...")
gene_ids = expr_filter.index.tolist()
res = mg.querymany(gene_ids, scopes=["ensembl.gene"], species="human", fields="symbol", returnall=False)
id_map = {item["query"]: item.get("symbol", item["query"]) for item in res}
expr_filter["symbol"] = expr_filter.index.map(id_map)
expr_final = expr_filter.dropna(subset=["symbol"]).groupby("symbol").mean()
print(f"最终基因数：{expr_final.shape[0]}")

# ---------------------- 4. 分别进行 limma 差异分析 ----------------------
# 提取各组样本名
control_samples = [gsm for gsm, g in SAMPLE_GROUP.items() if g == "Control"]
rankl_samples = [gsm for gsm, g in SAMPLE_GROUP.items() if g == "RANKL"]
tymp_samples = [gsm for gsm, g in SAMPLE_GROUP.items() if g == "TYMP"]

def run_limma(expr_sub, group_labels, design_name, contrast_name):
    """执行 limma 差异分析"""
    expr_log2 = np.log2(expr_sub + 1)
    # 设计矩阵
    design = pd.DataFrame({
        'Intercept': 1,
        contrast_name: [1 if g == contrast_name else 0 for g in group_labels]
    })
    with localconverter(ro.default_converter + pandas2ri.converter):
        ro.globalenv['expr_mat'] = expr_log2
        ro.globalenv['design_mat'] = design
        ro.r('''
            library(limma)
            fit <- lmFit(expr_mat, design_mat)
            fit <- eBayes(fit)
            res <- topTable(fit, coef=2, number=nrow(expr_mat), sort.by="none")
        ''')
        res_df = ro.globalenv['res']
    diff = pd.DataFrame({
        "symbol": expr_sub.index,
        "log2FoldChange": res_df['logFC'].values,
        "pvalue": res_df['P.Value'].values,
        "padj": res_df['adj.P.Val'].values,
    })
    diff["abs_log2FC"] = np.abs(diff["log2FoldChange"])
    return diff

# 4.1 RANKL vs Control
print("\n分析 RANKL vs Control...")
expr_rankl = expr_final[control_samples + rankl_samples]
group_rankl = ['Control']*3 + ['RANKL']*3
diff_rankl = run_limma(expr_rankl, group_rankl, "RANKL_vs_Control", "RANKL")
up_rankl = diff_rankl[(diff_rankl["padj"] < P_THRESHOLD) & (diff_rankl["log2FoldChange"] > LOG2FC_THRESHOLD)]
down_rankl = diff_rankl[(diff_rankl["padj"] < P_THRESHOLD) & (diff_rankl["log2FoldChange"] < -LOG2FC_THRESHOLD)]
print(f"  上调基因：{len(up_rankl)}，下调基因：{len(down_rankl)}")

# 4.2 TYMP vs Control
print("\n分析 TYMP vs Control...")
expr_tymp = expr_final[control_samples + tymp_samples]
group_tymp = ['Control']*3 + ['TYMP']*3
diff_tymp = run_limma(expr_tymp, group_tymp, "TYMP_vs_Control", "TYMP")
up_tymp = diff_tymp[(diff_tymp["padj"] < P_THRESHOLD) & (diff_tymp["log2FoldChange"] > LOG2FC_THRESHOLD)]
down_tymp = diff_tymp[(diff_tymp["padj"] < P_THRESHOLD) & (diff_tymp["log2FoldChange"] < -LOG2FC_THRESHOLD)]
print(f"  上调基因：{len(up_tymp)}，下调基因：{len(down_tymp)}")

# ---------------------- 5. 保存结果 ----------------------
# 原始表达矩阵（基因×样本）
expr_final.to_csv(os.path.join(SAVE_DIR, "GSE171542_原始表达矩阵(FPKM).csv"), index=True, encoding="utf-8-sig")

# RANKL vs Control 结果
diff_rankl.to_csv(os.path.join(SAVE_DIR, "GSE171542_差异分析结果_RANKL_vs_Control.csv"), index=False, encoding="utf-8-sig")
up_rankl.sort_values("log2FoldChange", ascending=False).to_csv(os.path.join(SAVE_DIR, "GSE171542_上调基因_RANKL_vs_Control.csv"), index=False, encoding="utf-8-sig")
down_rankl.sort_values("log2FoldChange", ascending=True).to_csv(os.path.join(SAVE_DIR, "GSE171542_下调基因_RANKL_vs_Control.csv"), index=False, encoding="utf-8-sig")

# TYMP vs Control 结果
diff_tymp.to_csv(os.path.join(SAVE_DIR, "GSE171542_差异分析结果_TYMP_vs_Control.csv"), index=False, encoding="utf-8-sig")
up_tymp.sort_values("log2FoldChange", ascending=False).to_csv(os.path.join(SAVE_DIR, "GSE171542_上调基因_TYMP_vs_Control.csv"), index=False, encoding="utf-8-sig")
down_tymp.sort_values("log2FoldChange", ascending=True).to_csv(os.path.join(SAVE_DIR, "GSE171542_下调基因_TYMP_vs_Control.csv"), index=False, encoding="utf-8-sig")

# 样本分组信息（更新为三组）
group_df = pd.DataFrame({
    "sample_id": expr_final.columns,
    "group": [SAMPLE_GROUP[col] for col in expr_final.columns]
})
group_df.to_csv(os.path.join(SAVE_DIR, "GSE171542_样本分组信息.csv"), index=False, encoding="utf-8-sig")

print("\n🎉 完成！分别使用 limma 分析了 RANKL vs Control 和 TYMP vs Control。")
print(f"结果已保存至：{SAVE_DIR}")

In [ ]:
# ===================== 火山图 + 热图绘制代码（适配 RANKL 和 TYMP 两个对比）=====================
import matplotlib.pyplot as plt
import seaborn as sns
from adjustText import adjust_text
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

# ===================== 路径配置（与差异分析代码保持一致）=====================
SAVE_DIR = r"./data"   # 请确保与前面代码的 SAVE_DIR 相同

# 绘图参数
P_THRESHOLD = 0.05
LOG2FC_THRESHOLD = 1.0

# 定义两个对比的名称及对应的差异文件
contrasts = [
    {
        "name": "RANKL_vs_Control",
        "diff_file": os.path.join(SAVE_DIR, "GSE171542_差异分析结果_RANKL_vs_Control.csv"),
        "title": "RANKL vs Control"
    },
    {
        "name": "TYMP_vs_Control",
        "diff_file": os.path.join(SAVE_DIR, "GSE171542_差异分析结果_TYMP_vs_Control.csv"),
        "title": "TYMP vs Control"
    }
]

# 读取表达矩阵和分组信息（两个对比共用）
expr_file = os.path.join(SAVE_DIR, "GSE171542_原始表达矩阵(FPKM).csv")
group_file = os.path.join(SAVE_DIR, "GSE171542_样本分组信息.csv")

expr_final = pd.read_csv(expr_file, index_col=0)
group_df = pd.read_csv(group_file)
GROUP_MAP = dict(zip(group_df["sample_id"], group_df["group"]))

# -----------------------------------------------------------------------------
# 循环处理每个对比
# -----------------------------------------------------------------------------
for contrast in contrasts:
    print(f"\n📌 处理对比：{contrast['title']}")
    diff_file = contrast["diff_file"]
    
    # 检查差异文件是否存在
    if not os.path.exists(diff_file):
        print(f"⚠️ 差异文件不存在，跳过：{diff_file}")
        continue
    
    diff_result = pd.read_csv(diff_file)
    # 确保 symbol 列作为索引（用于后续匹配）
    diff_result.set_index("symbol", inplace=True)
    
    # 过滤无效值
    diff_plot = diff_result.replace([np.inf, -np.inf], np.nan).dropna(subset=["log2FoldChange", "padj"])
    
    # 分类基因
    ns = diff_plot[~((diff_plot["padj"] < P_THRESHOLD) & (abs(diff_plot["log2FoldChange"]) > LOG2FC_THRESHOLD))]
    up = diff_plot[(diff_plot["padj"] < P_THRESHOLD) & (diff_plot["log2FoldChange"] > LOG2FC_THRESHOLD)]
    down = diff_plot[(diff_plot["padj"] < P_THRESHOLD) & (diff_plot["log2FoldChange"] < -LOG2FC_THRESHOLD)]
    
    print(f"  上调基因数：{len(up)}，下调基因数：{len(down)}")
    
    # -------------------------------------------------------------------------
    # 1. 火山图
    # -------------------------------------------------------------------------
    plt.figure(figsize=(10, 6))
    plt.scatter(ns["log2FoldChange"], -np.log10(ns["padj"]), c="gray", s=20, alpha=0.5, label="NS")
    if len(up) > 0:
        plt.scatter(up["log2FoldChange"], -np.log10(up["padj"]), c="#E63946", s=30, alpha=0.8, label=f"Up ({len(up)})")
    if len(down) > 0:
        plt.scatter(down["log2FoldChange"], -np.log10(down["padj"]), c="#457B9D", s=30, alpha=0.8, label=f"Down ({len(down)})")
    
    # 阈值线
    plt.axvline(x=LOG2FC_THRESHOLD, color="black", linestyle="--", lw=1, alpha=0.6)
    plt.axvline(x=-LOG2FC_THRESHOLD, color="black", linestyle="--", lw=1, alpha=0.6)
    plt.axhline(y=-np.log10(P_THRESHOLD), color="black", linestyle="--", lw=1, alpha=0.6)
    
    # 标注 top 10 差异基因（按 padj 排序，仅当有差异基因时）
    if len(up) + len(down) > 0:
        top10 = diff_plot.sort_values("padj").head(10)
        texts = []
        for idx, row in top10.iterrows():
            texts.append(plt.text(row["log2FoldChange"], -np.log10(row["padj"]), 
                                  idx[:18], fontsize=7))  # idx 是 gene symbol
        adjust_text(texts, arrowprops=dict(arrowstyle="->", color="black", alpha=0.3, lw=0.5))
    
    plt.xlabel("log2(Fold Change)", fontsize=12)
    plt.ylabel("-log10(Adjusted P-value)", fontsize=12)
    plt.title(f"GSE171542 Volcano Plot ({contrast['title']})", fontsize=14, fontweight="bold")
    plt.legend(loc="upper right", frameon=True)
    plt.grid(alpha=0.3)
    plt.xlim(-6, 6)
    plt.ylim(0, 6)
    
    volcano_path = os.path.join(SAVE_DIR, f"GSE171542_火山图_{contrast['name']}.png")
    plt.tight_layout()
    plt.savefig(volcano_path, dpi=300, bbox_inches="tight", facecolor="white")
    plt.close()
    print(f"✅ 火山图保存：{volcano_path}")
    
    # -------------------------------------------------------------------------
    # 2. 热图（仅当存在差异基因时绘制）
    # -------------------------------------------------------------------------
    if len(up) + len(down) == 0:
        print("⚠️ 无差异基因，跳过热图绘制")
        continue
    
    top_num = min(50, len(up) + len(down))
    # 按 padj 排序取 top 基因
    top_genes = diff_plot.sort_values("padj").head(top_num).index.tolist()
    # 提取表达矩阵子集（确保基因存在于表达矩阵中）
    available_genes = [g for g in top_genes if g in expr_final.index]
    if len(available_genes) == 0:
        print("⚠️ 热图：无可用基因，跳过")
        continue
    
    heat_data = expr_final.loc[available_genes]
    
    # Z-score 标准化
    heat_scaled = heat_data.sub(heat_data.mean(axis=1), axis=0).div(heat_data.std(axis=1), axis=0)
    
    # 样本分组颜色（根据 GROUP_MAP）
    group_colors = {"Control": "#A8DADC", "RANKL": "#FFB703", "TYMP": "#E9C46A"}
    col_colors = [group_colors[GROUP_MAP[sample]] for sample in heat_data.columns]
    
    # 聚类热图
    g = sns.clustermap(
        heat_scaled,
        cmap="RdBu_r", center=0,
        row_cluster=True, col_cluster=False,
        col_colors=col_colors,
        cbar_kws={"label": "Z-score"},
        linewidths=0.2,
        figsize=(11, 10),
        dendrogram_ratio=(0.1, 0.05)
    )
    g.ax_col_dendrogram.set_visible(False)
    g.fig.suptitle(f"Top {len(available_genes)} Differentially Expressed Genes\n({contrast['title']})", 
                   y=1.02, fontsize=14, fontweight="bold")
    
    heatmap_path = os.path.join(SAVE_DIR, f"GSE171542_热图_{contrast['name']}.png")
    plt.savefig(heatmap_path, dpi=300, bbox_inches="tight", facecolor="white")
    plt.close()
    print(f"✅ 热图保存：{heatmap_path}")

print("\n🎉 所有绘图任务完成！结果已保存至：", SAVE_DIR)



### GSE276597



In [ ]:


# ==============================================
# GSE276597 表达矩阵处理 【DESeq2 标准流程】
# 使用 DESeq2 进行差异分析（适用于原始整数 counts）
# 输出原始未标准化矩阵 | 支持 txt.gz 输入
# 路径已固定 | 直接运行 | 上调/下调基因输出
# 阈值：标准严格阈值 |log2FC|>1 + FDR<0.05
# ==============================================
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

# ---------------------- rpy2 相关导入 ----------------------
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri
from rpy2.robjects.conversion import localconverter
from rpy2.robjects.packages import importr, isinstalled

# 确保 DESeq2 已安装（自动安装）
utils = importr('utils')
if not isinstalled('DESeq2'):
    print("正在安装 DESeq2 ...")
    utils.install_packages('BiocManager')
    ro.r('BiocManager::install("DESeq2", update=FALSE, ask=FALSE)')
deseq2 = importr('DESeq2')

# ===================== 固定路径 =====================
RAW_DATA_PATH = r"./17 骨关节炎和囊泡/GEO 松动组织 vs. 稳定假体组织的基因表达谱/标本为人/GSE276597/GSE276597_Galaxy70-DESeq2_result_overview.txt.gz"
SAVE_DIR = r"./data"
# ==================================================================

os.makedirs(SAVE_DIR, exist_ok=True)

# ---------------------- 1. 读取原始数据 ----------------------
print("="*50)
print("读取 GSE276597 原始数据...")
print("="*50)
df_raw = pd.read_csv(RAW_DATA_PATH, sep="\t", compression="infer", low_memory=False)
print("文件列名（前10个）：", list(df_raw.columns)[:10])
print(f"数据形状：{df_raw.shape}")

# ---------------------- 2. 分离注释列和样本表达列 ----------------------
anno_cols = ["Sort0", "Gene ID", "Gene name", "Gene description", "Chr", 
             "Start", "End", "Strand", "Length", "IGV coordinates"]
sample_cols = [col for col in df_raw.columns if col not in anno_cols]
print(f"\n✅ 识别到样本列：{len(sample_cols)} 个")
print(f"样本列示例：{sample_cols[:5]}...")

# ---------------------- 3. 构建原始整数表达矩阵 ----------------------
expr_df = df_raw.set_index("Gene ID")[sample_cols].copy()
expr_df = expr_df.apply(pd.to_numeric, errors='coerce').fillna(0).astype(int)  # 转为整数
print(f"\n✅ 原始整数矩阵形状：{expr_df.shape}（基因 × 样本）")

# ---------------------- 4. 低表达过滤（基于原始 counts）---------------------
print("\n执行低表达基因过滤...")
threshold = max(2, expr_df.shape[1] * 0.3)  # 至少30%样本有表达
df_filter = expr_df.loc[(expr_df > 0).sum(axis=1) >= threshold]
print(f"✅ 过滤后矩阵形状：{df_filter.shape}")

# ---------------------- 5. 基因ID转换（Ensembl → Gene Symbol）并合并同基因计数 ----------------------
print("\n执行基因ID转换并合并同基因计数（求和）...")
id_map = df_raw.set_index("Gene ID")["Gene name"].to_dict()
df_filter["symbol"] = df_filter.index.map(id_map)
# 对同一基因的多个转录本进行 **求和**（保持整数 counts）
df_final = df_filter.dropna(subset=["symbol"]).groupby("symbol").sum()
print(f"✅ ID转换完成，最终基因数：{df_final.shape[0]}")

# ---------------------- 6. 样本分组 ----------------------
print("\n配置样本分组...")
GROUP_MAP = {
    "N4161":"Control","N4162":"Control","N4147":"Control","N4148":"Control","N4149":"Control",
    "N4150":"Control","N4151":"Control","N4157":"Control","N4158":"Control","N4159":"Control",
    "N4160":"Control","N4163":"Control","N4164":"Control","N4152":"Control","N4153":"Control",
    "N4154":"Control","N4155":"Control","N4156":"Control",
    "N5033":"Case","N5034":"Case","N5035":"Case","N5026":"Case","N5027":"Case",
    "N5028":"Case","N5029":"Case","N5030":"Case","N5031":"Case","N5032":"Case"
}
control_samples = [s for s, g in GROUP_MAP.items() if g == "Control"]
case_samples = [s for s, g in GROUP_MAP.items() if g == "Case"]
print(f"✅ 对照组(Control)样本数：{len(control_samples)}")
print(f"✅ 处理组(Case)样本数：{len(case_samples)}")

# ---------------------- 7. 差异分析（DESeq2 标准流程） ----------------------
print("\n开始差异分析（DESeq2，适用于原始整数 counts）...")

# 准备 colData（样本信息）
col_data = pd.DataFrame({
    "sample": df_final.columns,
    "condition": [GROUP_MAP[col] for col in df_final.columns]
})
col_data["condition"] = col_data["condition"].astype("category")
col_data = col_data.set_index("sample")

# 设计公式
design = ro.Formula('~ condition')

# 使用 rpy2 调用 DESeq2
with localconverter(ro.default_converter + pandas2ri.converter):
    ro.globalenv['counts_mat'] = df_final.astype(int)   # 确保整数
    ro.globalenv['col_data'] = col_data
    ro.globalenv['design'] = design
    
    ro.r('''
        library(DESeq2)
        dds <- DESeqDataSetFromMatrix(countData = counts_mat,
                                      colData = col_data,
                                      design = design)
        dds <- DESeq(dds)
        res <- results(dds, contrast = c("condition", "Case", "Control"))
        res_df <- as.data.frame(res)
    ''')
    
    res_df = ro.globalenv['res_df']

# 提取结果列（DESeq2 输出列名：log2FoldChange, pvalue, padj）
log2fc = res_df['log2FoldChange'].values
p_values = res_df['pvalue'].values
padj = res_df['padj'].values

# 处理 NA 值（某些基因可能因低表达被自动过滤，padj 为 NA）
padj = np.nan_to_num(padj, nan=1.0)
p_values = np.nan_to_num(p_values, nan=1.0)

# 构建差异结果表（与原代码格式一致）
diff_result = pd.DataFrame({
    "symbol": df_final.index,
    "log2FoldChange": log2fc,
    "pvalue": p_values,
    "padj": padj,
    "abs_log2FC": np.abs(log2fc)
})

# ---------------------- 8. 筛选差异基因 ----------------------
P_THRESHOLD = 0.05
LOG2FC_THRESHOLD = 1.0

print(f"\n筛选差异基因（阈值：padj < {P_THRESHOLD} 且 |log2FC| > {LOG2FC_THRESHOLD}）...")
up_genes = diff_result[(diff_result["padj"] < P_THRESHOLD) & (diff_result["log2FoldChange"] > LOG2FC_THRESHOLD)]
down_genes = diff_result[(diff_result["padj"] < P_THRESHOLD) & (diff_result["log2FoldChange"] < -LOG2FC_THRESHOLD)]

print(f"✅ 上调基因数（Case>Control）：{len(up_genes)}")
print(f"✅ 下调基因数（Case<Control）：{len(down_genes)}")
print(f"✅ 总差异基因数：{len(up_genes) + len(down_genes)}")

# ---------------------- 9. 保存结果文件 ----------------------
print("\n" + "="*50)
print("保存结果文件...")
print("="*50)

# 原始表达矩阵（整数 counts，用于热图等）
df_final.to_csv(os.path.join(SAVE_DIR, "GSE276597_原始表达矩阵.csv"), index=True, encoding="utf-8-sig")

# 完整差异分析结果
diff_result.to_csv(os.path.join(SAVE_DIR, "GSE276597_差异分析结果_Case_vs_Control.csv"), index=False, encoding="utf-8-sig")

# 上调基因
up_sorted = up_genes.sort_values("log2FoldChange", ascending=False)
up_sorted.to_csv(os.path.join(SAVE_DIR, "GSE276597_上调基因_Case_vs_Control.csv"), index=False, encoding="utf-8-sig")

# 下调基因
down_sorted = down_genes.sort_values("log2FoldChange", ascending=True)
down_sorted.to_csv(os.path.join(SAVE_DIR, "GSE276597_下调基因_Case_vs_Control.csv"), index=False, encoding="utf-8-sig")

# 样本分组信息
group_df = pd.DataFrame({"sample_id": df_final.columns, "group": [GROUP_MAP[s] for s in df_final.columns]})
group_df.to_csv(os.path.join(SAVE_DIR, "GSE276597_样本分组信息.csv"), index=False, encoding="utf-8-sig")

print("\n🎉🎉🎉 GSE276597 数据处理 + DESeq2 差异分析全部完成！")
print(f"📊 最终表达矩阵形状：{df_final.shape}（基因×样本）")
print(f"📈 上调基因数：{len(up_sorted)}，下调基因数：{len(down_sorted)}")
print(f"📂 所有结果已保存至：{SAVE_DIR}")



In [ ]:


# -*- coding: utf-8 -*-
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap

plt.rcParams['font.sans-serif'] = ['DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# ===================== 基础配置（与差异分析代码保持一致）=====================
SAVE_DIR = "./data/"
GSE_ID = "GSE276597"

# 阈值
FC_THRESH = 1
FDR_THRESH = 0.05

# ===================== 函数：绘制火山图 =====================
def plot_volcano(input_file, save_name, title_name):
    df = pd.read_csv(input_file, index_col=0)

    # 兼容列名：padj / FDR
    if 'padj' in df.columns:
        df.rename(columns={'padj': 'FDR'}, inplace=True)
    if 'log2FoldChange' in df.columns:
        df.rename(columns={'log2FoldChange': 'log2FC'}, inplace=True)

    df['-log10(FDR)'] = -np.log10(df['FDR'].astype(float))

    def classify(row):
        if row['FDR'] < FDR_THRESH and row['log2FC'] > FC_THRESH:
            return 'Up'
        elif row['FDR'] < FDR_THRESH and row['log2FC'] < -FC_THRESH:
            return 'Down'
        else:
            return 'No'

    df['type'] = df.apply(classify, axis=1)

    plt.figure(figsize=(6, 5), dpi=300)
    colors = {'Up': '#FF4C4C', 'Down': '#4C6EFF', 'No': '#B8B8B8'}
    sns.scatterplot(x='log2FC', y='-log10(FDR)', hue='type',
                    data=df, palette=colors, s=8, alpha=0.8, legend='full')

    plt.axvline(x=FC_THRESH, color='gray', linestyle='--', lw=0.8)
    plt.axvline(x=-FC_THRESH, color='gray', linestyle='--', lw=0.8)
    plt.axhline(y=-np.log10(FDR_THRESH), color='gray', linestyle='--', lw=0.8)

    plt.title(f'{GSE_ID} {title_name}', fontsize=12, pad=15)
    plt.xlabel('log2(Fold Change)', fontsize=10)
    plt.ylabel('-log10(FDR)', fontsize=10)
    plt.xlim(-6, 6)
    plt.legend(title='Gene Type', loc='upper right')
    plt.tight_layout()

    plt.savefig(f"{SAVE_DIR}{save_name}.png", dpi=300, bbox_inches='tight')
    plt.close()

    up_num = len(df[df['type'] == 'Up'])
    down_num = len(df[df['type'] == 'Down'])
    print(f"✅ {title_name} 火山图完成 | 上调:{up_num} | 下调:{down_num}")

# ===================== 热图函数 =====================
def plot_heatmap(matrix_file, group_file, diff_file, save_name, top_n=50):
    # 1. 读取数据
    expr = pd.read_csv(matrix_file, index_col=0)
    group = pd.read_csv(group_file)
    diff = pd.read_csv(diff_file)

    # 2. 筛选显著差异基因
    diff_sig = diff[
        (diff['padj'] < FDR_THRESH) &
        (abs(diff['log2FoldChange']) > FC_THRESH)
    ].copy()

    if len(diff_sig) == 0:
        print("⚠️ 无显著差异基因，跳过热图")
        return

    # 3. 取Top N差异基因
    diff_sig['abs_log2FC'] = diff_sig['log2FoldChange'].abs()
    top_genes = diff_sig.sort_values('abs_log2FC', ascending=False).head(top_n)['symbol']

    # 4. 提取矩阵 + Z-score标准化
    expr_plot = expr.loc[expr.index.isin(top_genes), :].copy()
    expr_plot = expr_plot.T.apply(lambda x: (x - x.mean()) / x.std()).T

    # 5. 清理基因名
    expr_plot.index = expr_plot.index.str.replace(r'\.\d+$', '', regex=True)

    # 6. 样本分组排序
    group_order = ['Case', 'Control']
    group['group_order'] = group['group'].map({g: i for i, g in enumerate(group_order)})
    sample_order = group.sort_values('group_order')['sample_id'].tolist()
    expr_plot = expr_plot[sample_order]

    # 7. 分组颜色条
    group_dict = dict(zip(group['sample_id'], group['group']))
    group_labels = [group_dict[s] for s in sample_order]
    group_colors_map = {'Case': '#1f77b4', 'Control': '#ff7f0e'}
    col_colors = [group_colors_map[g] for g in group_labels]

    # 8. 色标配置
    cmap = sns.diverging_palette(240, 10, as_cmap=True)
    vmin, vmax = -2.5, 2.5

    # 9. 绘制热图
    g = sns.clustermap(
        expr_plot,
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
        row_cluster=True,
        col_cluster=False,
        col_colors=col_colors,
        figsize=(12, 10),
        dendrogram_ratio=(0.1, 0.05),
        cbar_pos=(0.02, 0.8, 0.03, 0.18),
        yticklabels=True,
        xticklabels=True,
        linewidths=0.1
    )

    # 10. 标题
    g.fig.suptitle(
        f'{GSE_ID} Top{top_n} Differentially Expressed Genes (Case vs Control)',
        y=1.02,
        fontsize=14,
        fontweight='bold'
    )

    # 11. 分组图例
    for g_name, color in group_colors_map.items():
        g.ax_col_dendrogram.bar(0, 0, color=color, label=g_name, linewidth=0)
    g.ax_col_dendrogram.legend(
        loc='upper right',
        bbox_to_anchor=(1.15, 1),
        fontsize=10,
        title='Group'
    )

    # 12. 标签旋转
    plt.setp(g.ax_heatmap.get_xticklabels(), rotation=45, ha='right', fontsize=8)
    plt.setp(g.ax_heatmap.get_yticklabels(), fontsize=8)

    # 13. 保存
    plt.tight_layout()
    g.savefig(f"{SAVE_DIR}{save_name}.png", dpi=300, bbox_inches='tight', facecolor='white')
    plt.close()

    up_num = len(diff_sig[diff_sig['log2FoldChange'] > 0])
    down_num = len(diff_sig[diff_sig['log2FoldChange'] < 0])
    print(f"✅ 热图完成：{save_name} | 上调:{up_num} | 下调:{down_num} | Top{top_n}")

# ===================== 一键运行 =====================
if __name__ == "__main__":
    print("🚀 开始绘制 GSE276597 火山图 + 热图...\n")

    # 火山图（使用差异分析输出文件）
    plot_volcano(
        input_file=f"{SAVE_DIR}{GSE_ID}_差异分析结果_Case_vs_Control.csv",
        save_name=f"{GSE_ID}_火山图_Case_vs_Control",
        title_name="Volcano Plot (Case vs Control)"
    )

    # 热图（使用差异分析输出文件和原始表达矩阵、分组信息）
    plot_heatmap(
        matrix_file=f"{SAVE_DIR}{GSE_ID}_原始表达矩阵.csv",
        group_file=f"{SAVE_DIR}{GSE_ID}_样本分组信息.csv",
        diff_file=f"{SAVE_DIR}{GSE_ID}_差异分析结果_Case_vs_Control.csv",
        save_name=f"{GSE_ID}_热图_差异Top50",
        top_n=50
    )

    print(f"\n🎉 全部图片已保存至：{SAVE_DIR}")




    
### GSE284391



In [ ]:


# ==============================================
# HA vs OA 单细胞表达矩阵构建（内存安全版）
# 修复基因列表类型错误
# ==============================================
import pandas as pd
import numpy as np
import os
import gzip
import warnings
import scipy.io as sio
from scipy.sparse import csr_matrix, hstack, save_npz, load_npz

warnings.filterwarnings('ignore')

# ===================== 路径配置 =====================
MTX_PATH = r"./17 骨关节炎和囊泡/GEO 血友病滑膜 vs. 骨关节炎滑膜的差异基因/标本为人/GSE284391_matrix.mtx.gz"
FEATURES_PATH = r"./17 骨关节炎和囊泡/GEO 血友病滑膜 vs. 骨关节炎滑膜的差异基因/标本为人/GSE284391_features.tsv.gz"
BARCODES_PATH = r"./17 骨关节炎和囊泡/GEO 血友病滑膜 vs. 骨关节炎滑膜的差异基因/标本为人/GSE284391_barcodes.tsv.gz"

GSE152805_DIR = r"./17 骨关节炎和囊泡/GEO 血友病滑膜 vs. 骨关节炎滑膜的差异基因/标本为人/GSE152805_RAW"
SAVE_DIR = r"./data/HA_vs_OA_analysis"
os.makedirs(SAVE_DIR, exist_ok=True)

# ==============================================================================
# 步骤1：处理 HA 数据（保存稀疏矩阵 + 基因索引映射）
# ==============================================================================
print("="*60)
print("步骤1：处理 GSE284391 HA 数据 -> 稀疏矩阵")
print("="*60)

with gzip.open(MTX_PATH, 'rb') as f:
    mtx_data = sio.mmread(f)
print(f"原始矩阵形状: {mtx_data.shape}")

features = pd.read_csv(FEATURES_PATH, sep="\t", compression="gzip", header=None)
if features.shape[1] >= 3:
    features.columns = ["gene_id", "gene_symbol", "feature_type"]
else:
    features.columns = ["gene_id", "gene_symbol"]
gene_symbols = features["gene_symbol"].astype(str).tolist()  # 确保字符串

barcodes = pd.read_csv(BARCODES_PATH, sep="\t", compression="gzip", header=None)
cell_ids = barcodes[0].tolist()

print(f"基因数: {len(gene_symbols)}, 细胞数: {len(cell_ids)}")

# 确定方向（基因 × 细胞）
if mtx_data.shape[0] == len(gene_symbols) and mtx_data.shape[1] == len(cell_ids):
    ha_sparse = csr_matrix(mtx_data)
    print("矩阵方向: 基因 × 细胞")
elif mtx_data.shape[1] == len(gene_symbols) and mtx_data.shape[0] == len(cell_ids):
    ha_sparse = csr_matrix(mtx_data.T)
    print("矩阵方向: 细胞 × 基因（已转置）")
else:
    raise ValueError("维度不匹配")

# 去除重复基因名（保留第一个）
unique_idx = ~pd.Series(gene_symbols).duplicated().values
ha_sparse = ha_sparse[unique_idx, :]
gene_symbols_unique = [gene_symbols[i] for i in range(len(gene_symbols)) if unique_idx[i]]

# 去除全零基因
nonzero_rows = ha_sparse.getnnz(axis=1) > 0
ha_sparse = ha_sparse[nonzero_rows, :]
gene_symbols_final = [gene_symbols_unique[i] for i in range(len(gene_symbols_unique)) if nonzero_rows[i]]

# 保存 HA 稀疏矩阵和元数据
save_npz(os.path.join(SAVE_DIR, "GSE284391_HA_sparse.npz"), ha_sparse)
# 保存基因列表时强制为字符串
pd.Series(gene_symbols_final).astype(str).to_csv(os.path.join(SAVE_DIR, "GSE284391_HA_genes.csv"), index=False, header=False)
pd.Series(cell_ids).to_csv(os.path.join(SAVE_DIR, "GSE284391_HA_cells.csv"), index=False, header=False)
# 保存基因到索引的映射（确保键和值都是字符串）
ha_gene_to_idx = {str(gene): i for i, gene in enumerate(gene_symbols_final)}
pd.Series(ha_gene_to_idx).astype(str).to_csv(os.path.join(SAVE_DIR, "GSE284391_HA_gene_mapping.csv"))

print(f"✅ HA 稀疏矩阵已保存: {ha_sparse.shape[0]} 基因 × {ha_sparse.shape[1]} 细胞")
print(f"   非零元素数: {ha_sparse.nnz}")

# 释放临时大对象
del mtx_data, ha_sparse, gene_symbols, gene_symbols_unique, gene_symbols_final
import gc; gc.collect()

# ==============================================================================
# 步骤2：处理 OA 数据（保存稀疏矩阵 + 基因映射）
# ==============================================================================
print("\n" + "="*60)
print("步骤2：处理 GSE152805 OA 数据 -> 稀疏矩阵")
print("="*60)

matrix_files = [f for f in os.listdir(GSE152805_DIR) if f.endswith(".matrix.mtx.gz")]
print(f"发现 OA matrix 文件: {matrix_files}")

oa_sparse_list = []
oa_gene_list = []
oa_cell_list = []
total_cells = 0

for matrix_file in matrix_files:
    prefix = matrix_file.replace(".matrix.mtx.gz", "")
    matrix_path = os.path.join(GSE152805_DIR, matrix_file)
    genes_path = os.path.join(GSE152805_DIR, f"{prefix}.genes.tsv.gz")
    barcodes_path = os.path.join(GSE152805_DIR, f"{prefix}.barcodes.tsv.gz")
    
    if not (os.path.exists(genes_path) and os.path.exists(barcodes_path)):
        print(f"  跳过 {prefix}：缺少基因或细胞文件")
        continue
    
    print(f"  读取 {prefix} ...")
    with gzip.open(matrix_path, 'rb') as f:
        mtx = sio.mmread(f)
    print(f"    原始形状: {mtx.shape}")
    
    genes = pd.read_csv(genes_path, sep="\t", compression="gzip", header=None)
    # 基因符号取第二列（若存在），并转为字符串
    if genes.shape[1] >= 2:
        gene_symbols_oa = genes[1].astype(str).tolist()
    else:
        gene_symbols_oa = genes[0].astype(str).tolist()
    barcodes_oa = pd.read_csv(barcodes_path, sep="\t", compression="gzip", header=None)
    cell_ids_oa = [f"{prefix}_{b}" for b in barcodes_oa[0].tolist()]
    print(f"    基因数: {len(gene_symbols_oa)}, 细胞数: {len(cell_ids_oa)}")
    
    # 判断方向
    if mtx.shape[0] == len(gene_symbols_oa) and mtx.shape[1] == len(cell_ids_oa):
        sparse_mat = csr_matrix(mtx)
        print("    方向: 基因 × 细胞")
    elif mtx.shape[1] == len(gene_symbols_oa) and mtx.shape[0] == len(cell_ids_oa):
        sparse_mat = csr_matrix(mtx.T)
        print("    方向: 细胞 × 基因（已转置）")
    else:
        print(f"    警告：维度不匹配，跳过")
        continue
    
    # 去重基因
    unique_idx = ~pd.Series(gene_symbols_oa).duplicated().values
    sparse_mat = sparse_mat[unique_idx, :]
    gene_symbols_oa_unique = [gene_symbols_oa[i] for i in range(len(gene_symbols_oa)) if unique_idx[i]]
    
    # 去除全零基因
    nonzero_rows = sparse_mat.getnnz(axis=1) > 0
    sparse_mat = sparse_mat[nonzero_rows, :]
    gene_symbols_oa_final = [gene_symbols_oa_unique[i] for i in range(len(gene_symbols_oa_unique)) if nonzero_rows[i]]
    
    oa_sparse_list.append(sparse_mat)
    oa_gene_list.append(gene_symbols_oa_final)
    oa_cell_list.append(cell_ids_oa)
    total_cells += len(cell_ids_oa)
    print(f"      保留细胞: {len(cell_ids_oa)}，累计 {total_cells}")

if not oa_sparse_list:
    raise RuntimeError("没有成功读取任何 OA 样本")

# 合并 OA 样本：基因交集
print("  合并 OA 样本（基因交集）...")
common_genes_oa = set(oa_gene_list[0])
for g in oa_gene_list[1:]:
    common_genes_oa &= set(g)
common_genes_oa = sorted(common_genes_oa)  # 此时全是字符串，可排序
print(f"    共同基因数: {len(common_genes_oa)}")

# 为每个样本构建基因到索引的字典，然后子集化
oa_sub_matrices = []
for mat, genes in zip(oa_sparse_list, oa_gene_list):
    gene_to_idx = {str(g): i for i, g in enumerate(genes)}  # 确保字符串键
    idx = [gene_to_idx[g] for g in common_genes_oa]
    oa_sub_matrices.append(mat[idx, :])
    del gene_to_idx
oa_combined = hstack(oa_sub_matrices)
oa_cells_combined = [cell for cells in oa_cell_list for cell in cells]

# 保存 OA 稀疏矩阵及映射
save_npz(os.path.join(SAVE_DIR, "GSE152805_OA_sparse.npz"), oa_combined)
pd.Series(common_genes_oa).astype(str).to_csv(os.path.join(SAVE_DIR, "GSE152805_OA_genes.csv"), index=False, header=False)
pd.Series(oa_cells_combined).to_csv(os.path.join(SAVE_DIR, "GSE152805_OA_cells.csv"), index=False, header=False)
oa_gene_to_idx = {str(gene): i for i, gene in enumerate(common_genes_oa)}
pd.Series(oa_gene_to_idx).astype(str).to_csv(os.path.join(SAVE_DIR, "GSE152805_OA_gene_mapping.csv"))

print(f"✅ OA 稀疏矩阵已保存: {oa_combined.shape[0]} 基因 × {oa_combined.shape[1]} 细胞")
print(f"   非零元素数: {oa_combined.nnz}")

# 释放临时 OA 列表和矩阵
del oa_sparse_list, oa_gene_list, oa_cell_list, oa_sub_matrices, oa_combined
gc.collect()

# ==============================================================================
# 步骤3：合并 HA 和 OA（使用字典映射，内存高效）
# ==============================================================================
print("\n" + "="*60)
print("步骤3：合并 HA 和 OA 稀疏矩阵（基因交集）")
print("="*60)

# 加载 HA 数据（指定 dtype=str 防止数字基因名变成 float）
ha_gene_mapping = pd.read_csv(
    os.path.join(SAVE_DIR, "GSE284391_HA_gene_mapping.csv"),
    index_col=0, header=None, dtype=str
).to_dict()[1]
# 过滤掉可能的 NaN 字符串（'nan'）
ha_gene_mapping = {k: v for k, v in ha_gene_mapping.items() if k != 'nan' and pd.notna(k)}
ha_genes = list(ha_gene_mapping.keys())
ha_cells = pd.read_csv(os.path.join(SAVE_DIR, "GSE284391_HA_cells.csv"), header=None, dtype=str)[0].tolist()
ha_sparse = load_npz(os.path.join(SAVE_DIR, "GSE284391_HA_sparse.npz"))

# 加载 OA 数据
oa_gene_mapping = pd.read_csv(
    os.path.join(SAVE_DIR, "GSE152805_OA_gene_mapping.csv"),
    index_col=0, header=None, dtype=str
).to_dict()[1]
oa_gene_mapping = {k: v for k, v in oa_gene_mapping.items() if k != 'nan' and pd.notna(k)}
oa_genes = list(oa_gene_mapping.keys())
oa_cells = pd.read_csv(os.path.join(SAVE_DIR, "GSE152805_OA_cells.csv"), header=None, dtype=str)[0].tolist()
oa_sparse = load_npz(os.path.join(SAVE_DIR, "GSE152805_OA_sparse.npz"))

# 基因交集（全部转为字符串后取交集）
ha_set = set(ha_genes)
oa_set = set(oa_genes)
common_genes = sorted(ha_set & oa_set)  # 现在都是字符串，可排序
print(f"共同基因数: {len(common_genes)}")

# 使用映射快速获取索引（O(1)）
ha_idx = [int(ha_gene_mapping[g]) for g in common_genes]  # 映射值是索引，转为 int
oa_idx = [int(oa_gene_mapping[g]) for g in common_genes]

# 子集化（保持稀疏）
ha_sub = ha_sparse[ha_idx, :]
oa_sub = oa_sparse[oa_idx, :]

# 释放原始大矩阵，减少内存峰值
del ha_sparse, oa_sparse
gc.collect()

# 合并（横向拼接）
combined_sparse = hstack([ha_sub, oa_sub])
combined_cells = ha_cells + oa_cells
combined_groups = ["HA"] * len(ha_cells) + ["OA"] * len(oa_cells)

# 保存最终结果
save_npz(os.path.join(SAVE_DIR, "HA_vs_OA_combined_sparse.npz"), combined_sparse)
pd.Series(common_genes).to_csv(os.path.join(SAVE_DIR, "HA_vs_OA_combined_genes.csv"), index=False, header=False)
pd.Series(combined_cells).to_csv(os.path.join(SAVE_DIR, "HA_vs_OA_combined_cells.csv"), index=False, header=False)
group_df = pd.DataFrame({"cell_id": combined_cells, "group": combined_groups})
group_df.to_csv(os.path.join(SAVE_DIR, "HA_vs_OA_group_info.csv"), index=False)

print(f"\n🎉 合并完成！")
print(f"📁 合并矩阵形状: {combined_sparse.shape[0]} 基因 × {combined_sparse.shape[1]} 细胞")
print(f"📁 HA 细胞数: {len(ha_cells)}")
print(f"📁 OA 细胞数: {len(oa_cells)}")
print(f"📁 非零元素数: {combined_sparse.nnz}")



In [ ]:


# ==============================================
# HA vs OA 单细胞差异分析 + 火山图 + 热图（最终稳定版）
# 修正：热图返回字典的处理
# ==============================================
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.sparse import load_npz
import scanpy as sc
from matplotlib import rcParams

# 设置 scanpy 输出详细进度
sc.settings.verbosity = 3

# 设置绘图风格
rcParams['figure.figsize'] = (8, 6)
rcParams['font.size'] = 10
sns.set_style("whitegrid")

# ===================== 路径配置 =====================
SAVE_DIR = r"./data/HA_vs_OA_analysis"
os.makedirs(SAVE_DIR, exist_ok=True)

# ===================== 1. 加载数据 =====================
print("加载稀疏矩阵和元数据...")
X = load_npz(os.path.join(SAVE_DIR, "HA_vs_OA_combined_sparse.npz"))
genes = pd.read_csv(os.path.join(SAVE_DIR, "HA_vs_OA_combined_genes.csv"), header=None)[0].tolist()
cells = pd.read_csv(os.path.join(SAVE_DIR, "HA_vs_OA_combined_cells.csv"), header=None)[0].tolist()
group_info = pd.read_csv(os.path.join(SAVE_DIR, "HA_vs_OA_group_info.csv"))

# 确保顺序一致
group_info = group_info.set_index("cell_id").loc[cells]
print(f"矩阵形状: {X.shape} 基因 × 细胞")
print(f"HA 细胞数: {(group_info['group'] == 'HA').sum()}")
print(f"OA 细胞数: {(group_info['group'] == 'OA').sum()}")

# ===================== 2. 构建 AnnData 对象 =====================
adata = sc.AnnData(X=X.T.astype(np.float32),
                   obs=group_info,
                   var=pd.DataFrame(index=genes))
print("AnnData 构建完成")

# ===================== 3. 质量控制与标准化 =====================
sc.pp.filter_cells(adata, min_genes=200)
sc.pp.filter_genes(adata, min_cells=3)

sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

# ===================== 4. 差异分析 =====================
print("开始差异分析...")
sc.tl.rank_genes_groups(adata, groupby='group', groups=['OA'], reference='HA',
                        method='wilcoxon', use_raw=False, n_jobs=4)

# 提取结果
de_df = sc.get.rank_genes_groups_df(adata, group='OA')
print("原始列名:", de_df.columns.tolist())

# 根据列名智能重命名
if 'logfoldchanges' in de_df.columns:
    de_df = de_df.rename(columns={
        'names': 'gene',
        'logfoldchanges': 'log2FC',
        'pvals': 'p_val',
        'pvals_adj': 'p_val_adj'
    })
    if 'scores' in de_df.columns:
        de_df = de_df.drop(columns=['scores'])
else:
    de_df.columns = ['gene', 'scores', 'log2FC', 'p_val', 'p_val_adj']
    de_df = de_df.drop(columns=['scores'])

de_df = de_df.sort_values('p_val_adj')
de_df.to_csv(os.path.join(SAVE_DIR, "HA_vs_OA_diff_all.csv"), index=False)
print(f"差异分析完成，共 {len(de_df)} 个基因")

# 筛选显著差异基因
sig_up = de_df[(de_df['log2FC'] > 1) & (de_df['p_val_adj'] < 0.05)]
sig_down = de_df[(de_df['log2FC'] < -1) & (de_df['p_val_adj'] < 0.05)]
sig_all = pd.concat([sig_up, sig_down])
print(f"显著上调基因: {len(sig_up)}")
print(f"显著下调基因: {len(sig_down)}")

# 分别保存上调/下调基因
sig_up.to_csv(os.path.join(SAVE_DIR, "HA_vs_OA_upregulated_genes.csv"), index=False)
sig_down.to_csv(os.path.join(SAVE_DIR, "HA_vs_OA_downregulated_genes.csv"), index=False)
sig_all.to_csv(os.path.join(SAVE_DIR, "HA_vs_OA_significant_genes.csv"), index=False)

# ===================== 5. 火山图 =====================
fig, ax = plt.subplots(figsize=(10, 8))
ax.scatter(de_df['log2FC'], -np.log10(de_df['p_val_adj']),
           c='lightgray', alpha=0.5, s=5, label='Not significant')
ax.scatter(sig_up['log2FC'], -np.log10(sig_up['p_val_adj']),
           c='red', alpha=0.8, s=20, label=f'Upregulated ({len(sig_up)})')
ax.scatter(sig_down['log2FC'], -np.log10(sig_down['p_val_adj']),
           c='blue', alpha=0.8, s=20, label=f'Downregulated ({len(sig_down)})')
ax.axhline(-np.log10(0.05), linestyle='--', color='black', alpha=0.5)
ax.axvline(-1, linestyle='--', color='black', alpha=0.5)
ax.axvline(1, linestyle='--', color='black', alpha=0.5)
ax.set_xlabel('log2 Fold Change (OA vs HA)', fontsize=12)
ax.set_ylabel('-log10(Adjusted P-value)', fontsize=12)
ax.set_title('Volcano Plot: OA vs HA', fontsize=14)
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "Volcano_plot.png"), dpi=300)
plt.show()
print("火山图已保存")

# ===================== 6. 热图（Top 20 上调 + Top 20 下调）- 单图版 =====================
print("识别高变基因（用于热图）...")
sc.pp.highly_variable_genes(adata, n_top_genes=2000, flavor='seurat', batch_key=None)

top_up = sig_up.nlargest(20, 'log2FC')['gene'].tolist()
top_down = sig_down.nsmallest(20, 'log2FC')['gene'].tolist()
top_genes = top_up + top_down
print(f"热图展示基因数: {len(top_genes)}")

# 下采样细胞
np.random.seed(42)
n_cells_sample = 200
ha_cells = adata.obs[adata.obs['group'] == 'HA'].index
oa_cells = adata.obs[adata.obs['group'] == 'OA'].index
if len(ha_cells) > n_cells_sample:
    ha_cells = np.random.choice(ha_cells, n_cells_sample, replace=False)
if len(oa_cells) > n_cells_sample:
    oa_cells = np.random.choice(oa_cells, n_cells_sample, replace=False)
sample_cells = np.concatenate([ha_cells, oa_cells])

adata_sub = adata[sample_cells, top_genes].copy()
adata_sub.obs['group'] = pd.Categorical(adata_sub.obs['group'], categories=['HA', 'OA'])
adata_sub = adata_sub[adata_sub.obs.sort_values('group').index]

# 直接绘制（不设置 show=False，让 scanpy 正常显示）
sc.pl.heatmap(adata_sub, var_names=top_genes, groupby='group',
              cmap='RdBu_r', standard_scale='var', 
              figsize=(12, 10), show_gene_labels=True,
              dendrogram=False, swap_axes=False)

# 获取当前图形并添加标题
fig = plt.gcf()
fig.suptitle('Top 20 Up/Down-regulated Genes (OA vs HA)', fontsize=16)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "Heatmap_top_genes.png"), dpi=300)
plt.close()
print(f"热图已保存至: {os.path.join(SAVE_DIR, 'Heatmap_top_genes.png')}")

# 保存表达矩阵
top_expr = pd.DataFrame(adata_sub.X.toarray(),
                        index=adata_sub.obs_names,
                        columns=top_genes)
top_expr.to_csv(os.path.join(SAVE_DIR, "Top_genes_expression_matrix.csv"))

print("\n🎉 所有分析完成！")
print(f"结果保存在: {SAVE_DIR}")
print("文件列表：")
print("  - HA_vs_OA_diff_all.csv              (全部基因差异结果)")
print("  - HA_vs_OA_upregulated_genes.csv     (上调基因)")
print("  - HA_vs_OA_downregulated_genes.csv   (下调基因)")
print("  - HA_vs_OA_significant_genes.csv     (显著差异基因)")
print("  - Volcano_plot.png                   (火山图)")
print("  - Heatmap_top_genes.png              (热图)")



### 人 关键基因 韦恩图

In [ ]:
# ==============================================
# 多个数据集差异基因整合与韦恩图
# 输入：各数据集的上调/下调基因CSV
# 输出：各数据集差异基因并集、四个数据集总并集、与HA_vs_OA交集、韦恩图
# ==============================================
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
from matplotlib_venn import venn2

# ===================== 路径配置 =====================
BASE_DIR = Path("./data")
HA_DIR = Path("./data/HA_vs_OA_analysis")

OUTPUT_DIR = Path("human_core_genes")
OUTPUT_DIR.mkdir(exist_ok=True)

# ===================== 通用读取函数 =====================
def get_diff_genes(up_file, down_file, gene_col):
    """读取上调下调文件，返回基因符号集合（并集），基因名统一转为大写"""
    up = pd.read_csv(up_file)
    down = pd.read_csv(down_file)
    up_genes = set(up[gene_col].dropna().astype(str).str.upper())
    down_genes = set(down[gene_col].dropna().astype(str).str.upper())
    return up_genes.union(down_genes)

# ===================== 1. GSE104589 =====================
print("="*50)
print("读取 GSE104589 差异基因...")
gse104589_up = BASE_DIR / "GSE104589_上调基因.csv"
gse104589_down = BASE_DIR / "GSE104589_下调基因.csv"
genes_104589 = get_diff_genes(gse104589_up, gse104589_down, gene_col="gene_id")
print(f"GSE104589 差异基因数: {len(genes_104589)}")

# ===================== 2. GSE149315 (circRNA宿主基因) =====================
print("\n读取 GSE149315 差异基因（已转换宿主基因）...")
gse149315_up = BASE_DIR / "GSE149315_显著上调circRNA_GeneSymbol.csv"
gse149315_down = BASE_DIR / "GSE149315_显著下调circRNA_GeneSymbol.csv"
up_149 = pd.read_csv(gse149315_up)
down_149 = pd.read_csv(gse149315_down)
genes_149315 = set(up_149['gene_symbol'].dropna().astype(str).str.upper()) | \
               set(down_149['gene_symbol'].dropna().astype(str).str.upper())
print(f"GSE149315 差异基因数: {len(genes_149315)}")

# ===================== 3. GSE171542（修改：取 RANKL 和 TYMP 两个对比的并集）=====================
print("\n读取 GSE171542 差异基因（RANKL vs Control 和 TYMP vs Control 的并集）...")

# RANKL vs Control 的差异基因
rankl_up = BASE_DIR / "GSE171542_上调基因_RANKL_vs_Control.csv"
rankl_down = BASE_DIR / "GSE171542_下调基因_RANKL_vs_Control.csv"
genes_rankl = get_diff_genes(rankl_up, rankl_down, gene_col="symbol")

# TYMP vs Control 的差异基因
tymp_up = BASE_DIR / "GSE171542_上调基因_TYMP_vs_Control.csv"
tymp_down = BASE_DIR / "GSE171542_下调基因_TYMP_vs_Control.csv"
genes_tymp = get_diff_genes(tymp_up, tymp_down, gene_col="symbol")

# 取两个对比的并集作为 GSE171542 的最终差异基因
genes_171542 = genes_rankl.union(genes_tymp)
print(f"  RANKL vs Control 差异基因数: {len(genes_rankl)}")
print(f"  TYMP vs Control 差异基因数: {len(genes_tymp)}")
print(f"  GSE171542 总差异基因数（并集）: {len(genes_171542)}")

# ===================== 4. GSE276597 =====================
print("\n读取 GSE276597 差异基因...")
gse276597_up = BASE_DIR / "GSE276597_上调基因_Case_vs_Control.csv"
gse276597_down = BASE_DIR / "GSE276597_下调基因_Case_vs_Control.csv"
genes_276597 = get_diff_genes(gse276597_up, gse276597_down, gene_col="symbol")
print(f"GSE276597 差异基因数: {len(genes_276597)}")

# ===================== 5. HA_vs_OA =====================
print("\n读取 HA_vs_OA 显著差异基因...")
ha_file = HA_DIR / "HA_vs_OA_significant_genes.csv"
ha_df = pd.read_csv(ha_file)
genes_ha = set(ha_df['gene'].dropna().astype(str).str.upper())
print(f"HA_vs_OA 显著差异基因数: {len(genes_ha)}")

# ===================== 6. 四个数据集的总并集 =====================
union_four = genes_104589 | genes_149315 | genes_171542 | genes_276597
print(f"\n四个数据集差异基因总并集: {len(union_four)}")

# 保存各数据集差异基因列表及总并集
pd.DataFrame({"Gene": sorted(genes_104589)}).to_csv(OUTPUT_DIR / "GSE104589_diff_genes.csv", index=False)
pd.DataFrame({"Gene": sorted(genes_149315)}).to_csv(OUTPUT_DIR / "GSE149315_diff_genes.csv", index=False)
pd.DataFrame({"Gene": sorted(genes_171542)}).to_csv(OUTPUT_DIR / "GSE171542_diff_genes.csv", index=False)
pd.DataFrame({"Gene": sorted(genes_276597)}).to_csv(OUTPUT_DIR / "GSE276597_diff_genes.csv", index=False)
pd.DataFrame({"Gene": sorted(union_four)}).to_csv(OUTPUT_DIR / "four_datasets_union_genes.csv", index=False)

# ===================== 7. 与 HA_vs_OA 取交集 =====================
common_genes = union_four & genes_ha
print(f"四个数据集并集与 HA_vs_OA 交集基因数: {len(common_genes)}")
pd.DataFrame({"Gene": sorted(common_genes)}).to_csv(OUTPUT_DIR / "human_core_genes_intersection.csv", index=False)

# ===================== 8. 韦恩图 =====================
plt.figure(figsize=(8, 6))
venn2(subsets=(len(union_four - common_genes),
               len(genes_ha - common_genes),
               len(common_genes)),
      set_labels=('Four datasets (union)', 'HA_vs_OA'))
plt.title('Overlap of Differential Genes', fontsize=14)
plt.savefig(OUTPUT_DIR / "Venn_union_four_vs_HA.png", dpi=300, bbox_inches='tight')
plt.show()

print("\n🎉 整合完成！所有输出文件保存在：", OUTPUT_DIR)
print("文件列表：")
print("  - GSE104589_diff_genes.csv")
print("  - GSE149315_diff_genes.csv")
print("  - GSE171542_diff_genes.csv")
print("  - GSE276597_diff_genes.csv")
print("  - four_datasets_union_genes.csv")
print("  - human_core_genes_intersection.csv")
print("  - Venn_union_four_vs_HA.png")



## 鼠
### GSE234863



In [ ]:


# ==============================================
# GSE234863 差异表达分析（DESeq2 标准流程）
# 修正：归一化矩阵转换为 DataFrame
# ==============================================
import pandas as pd
import numpy as np
import os
import re
import warnings
warnings.filterwarnings('ignore')

# rpy2 相关
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri
from rpy2.robjects.conversion import localconverter
from rpy2.robjects.packages import importr, isinstalled

# 自动安装 DESeq2 依赖（若未安装）
utils = importr('utils')
if not isinstalled('BiocManager'):
    utils.install_packages('BiocManager', quiet=True)
if not isinstalled('DESeq2'):
    print("正在安装 DESeq2 ...")
    ro.r('BiocManager::install("DESeq2", update=FALSE, ask=FALSE)')

# ===================== 路径配置 =====================
RAW_DATA_PATH = r"./17 骨关节炎和囊泡/GEO 松动组织 vs. 稳定假体组织的基因表达谱/标本为鼠/GSE234863_all.genes.expression.annot.txt.gz"
SAVE_DIR = r"./data"
os.makedirs(SAVE_DIR, exist_ok=True)

# ===================== 实验设计 =====================
CONTROL_GROUP = "Naive"
CASE_GROUP    = "Ti"
P_THRESHOLD   = 0.05
LOG2FC_THRESHOLD = 0.585  # 相当于 1.5 倍变化（log2(1.5)≈0.585）

# ===================== 1. 读取数据 =====================
print("="*50)
print("读取 GSE234863 原始数据...")
print("="*50)
df_raw = pd.read_csv(RAW_DATA_PATH, sep="\t", compression="infer", low_memory=False)
print(f"原始形状：{df_raw.shape}")

# 识别注释列
anno_keywords = ["id", "Symbol", "Description", "KEGG", "Pathway", "GO", "TF", "Chr", "Start", "End", "Strand", "Length"]
anno_cols = [col for col in df_raw.columns if any(kw in col for kw in anno_keywords)]
all_sample_cols = [col for col in df_raw.columns if col not in anno_cols]

# 只保留 _count 列
count_cols = [col for col in all_sample_cols if col.endswith("_count")]
print(f"找到 counts 列：{count_cols}")

if len(count_cols) == 0:
    raise ValueError("未找到 _count 列，请检查数据。")

# 构建 count matrix（基因 × 样本）
count_df = df_raw.set_index("id")[count_cols].copy()
count_df = count_df.apply(pd.to_numeric, errors='coerce').fillna(0).astype(int)

# 提取 Symbol 映射
if "Symbol" in df_raw.columns:
    symbol_series = df_raw.set_index("id")["Symbol"]
else:
    symbol_series = pd.Series(count_df.index, index=count_df.index)

print(f"Count 矩阵形状：{count_df.shape}（基因 × 样本）")

# ===================== 2. 样本分组 =====================
GROUP_MAP = {}
for col in count_cols:
    base = col.replace("_count", "")
    group = base.split("-")[0]
    GROUP_MAP[col] = group

control_samples = [s for s, g in GROUP_MAP.items() if g == CONTROL_GROUP]
case_samples = [s for s, g in GROUP_MAP.items() if g == CASE_GROUP]
print(f"对照组 {CONTROL_GROUP} 样本：{control_samples}")
print(f"实验组 {CASE_GROUP} 样本：{case_samples}")

sample_order = control_samples + case_samples
count_df = count_df[sample_order]
condition = [CONTROL_GROUP] * len(control_samples) + [CASE_GROUP] * len(case_samples)

# ===================== 3. DESeq2 分析 =====================
print("\n执行 DESeq2 分析...")
with localconverter(ro.default_converter + pandas2ri.converter):
    ro.globalenv['count_matrix'] = count_df
    ro.globalenv['condition'] = condition
    ro.r('''
        library(DESeq2)
        colData <- data.frame(condition = factor(condition, levels=c("Naive","Ti")))
        dds <- DESeqDataSetFromMatrix(countData = count_matrix,
                                      colData = colData,
                                      design = ~ condition)
        dds <- DESeq(dds)
        res <- results(dds, contrast=c("condition", "Ti", "Naive"))
        res_df <- as.data.frame(res)
        # 提取归一化计数并赋值给变量
        norm_counts <- counts(dds, normalized=TRUE)
    ''')
    res_df = ro.globalenv['res_df']
    norm_counts = ro.globalenv['norm_counts']

# 转换回 pandas DataFrame
with localconverter(ro.default_converter + pandas2ri.converter):
    res_pd = ro.conversion.rpy2py(res_df)
    norm_counts_array = ro.conversion.rpy2py(norm_counts)   # 这是 numpy 数组，形状 (基因数, 样本数)

# ===================== 4. 构建归一化表达矩阵 DataFrame =====================
# norm_counts_array 是基因×样本的矩阵，行名和列名需要从 count_df 获取
gene_ids = count_df.index.tolist()   # 基因 ID 顺序与 DESeq2 输入一致
sample_names = sample_order           # 样本顺序与输入一致
norm_pd = pd.DataFrame(norm_counts_array, index=gene_ids, columns=sample_names)

# ===================== 5. 整理差异结果 =====================
res_pd = res_pd.reset_index().rename(columns={"index": "id"})
res_pd["Symbol"] = res_pd["id"].map(symbol_series)

required_cols = ["id", "Symbol", "baseMean", "log2FoldChange", "lfcSE", "stat", "pvalue", "padj"]
for col in required_cols:
    if col not in res_pd.columns:
        res_pd[col] = np.nan

res_pd["padj"] = res_pd["padj"].fillna(1.0)
res_pd["pvalue"] = res_pd["pvalue"].fillna(1.0)

sig = (res_pd["padj"] < P_THRESHOLD) & (abs(res_pd["log2FoldChange"]) > LOG2FC_THRESHOLD)
up_genes = res_pd[sig & (res_pd["log2FoldChange"] > 0)].sort_values("log2FoldChange", ascending=False)
down_genes = res_pd[sig & (res_pd["log2FoldChange"] < 0)].sort_values("log2FoldChange", ascending=True)

print(f"\n✅ 上调基因数：{len(up_genes)}")
print(f"✅ 下调基因数：{len(down_genes)}")
print(f"✅ 总差异基因数：{len(up_genes) + len(down_genes)}")

# ===================== 6. 保存结果 =====================
gse_id = "GSE234863"

res_pd.to_csv(os.path.join(SAVE_DIR, f"{gse_id}_DESeq2_full_results.csv"), index=False)
up_genes.to_csv(os.path.join(SAVE_DIR, f"{gse_id}_DESeq2_upregulated.csv"), index=False)
down_genes.to_csv(os.path.join(SAVE_DIR, f"{gse_id}_DESeq2_downregulated.csv"), index=False)

# 保存归一化表达矩阵（已是基因×样本，无需转置）
norm_pd.to_csv(os.path.join(SAVE_DIR, f"{gse_id}_DESeq2_normalized_counts.csv"))

group_info = pd.DataFrame({"sample_id": sample_order, "group": condition})
group_info.to_csv(os.path.join(SAVE_DIR, f"{gse_id}_sample_groups.csv"), index=False)

print("\n🎉 DESeq2 分析完成！")
print(f"📊 分析矩阵：{count_df.shape[0]} 基因 × {count_df.shape[1]} 样本")
print(f"📂 输出目录：{SAVE_DIR}")
print("输出文件：")
print("  - GSE234863_DESeq2_full_results.csv")
print("  - GSE234863_DESeq2_upregulated.csv")
print("  - GSE234863_DESeq2_downregulated.csv")
print("  - GSE234863_DESeq2_normalized_counts.csv")
print("  - GSE234863_sample_groups.csv")



In [ ]:
# ==============================================
# 火山图 + 基因表达热图 绘制代码（适配 DESeq2 输出）
# 基于 GSE234863 DESeq2 分析结果
# 标题使用英文，避免中文字体缺失
# ==============================================
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap

# ---------------------- 全局绘图设置（修复中文乱码） ----------------------
plt.rcParams['font.sans-serif'] = ['WenQuanYi Zen Hei', 'DejaVu Sans', 'Arial Unicode MS', 'SimHei']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 300
plt.rcParams['font.size'] = 10
sns.set_style("white")

# ===================== 配置（与 DESeq2 分析一致） =====================
SAVE_DIR = r"./data"
GSE_ID = "GSE234863"
CONTROL_GROUP = "Naive"
CASE_GROUP = "Ti"

# 绘图阈值
P_CUTOFF = 0.05
FC_CUTOFF = 1.0          # log2FoldChange 阈值（2倍变化）
TOP_N_HEATMAP = 50       # 热图展示前 N 个差异基因

# 文件路径（DESeq2 输出）
diff_file = os.path.join(SAVE_DIR, f"{GSE_ID}_DESeq2_full_results.csv")
expr_file = os.path.join(SAVE_DIR, f"{GSE_ID}_DESeq2_normalized_counts.csv")
group_file = os.path.join(SAVE_DIR, f"{GSE_ID}_sample_groups.csv")

# ==============================================================================
# 一、读取数据
# ==============================================================================
print("="*50)
print("读取 DESeq2 差异分析结果、归一化表达矩阵及分组信息...")
print("="*50)

diff_result = pd.read_csv(diff_file)
expr_norm = pd.read_csv(expr_file, index_col=0)   # 行为基因 ID (Ensembl ID)，列为样本
group_df = pd.read_csv(group_file)

print(f"✅ 差异结果基因数：{diff_result.shape[0]}")
print(f"✅ 归一化表达矩阵形状：{expr_norm.shape}")
print(f"✅ 样本分组信息：{group_df.shape[0]} 个样本")

# 提取样本列表（按分组文件中的顺序）
sample_cols = group_df['sample_id'].tolist()
# 确保表达矩阵的列顺序与样本列表一致
expr_norm = expr_norm[sample_cols]

# 构建样本分组字典（用于热图颜色条，可选）
group_map = dict(zip(group_df['sample_id'], group_df['group']))

# ==============================================================================
# 二、火山图（标题英文）
# ==============================================================================
print("\n" + "="*50)
print("开始绘制火山图...")
print("="*50)

# 定义分类函数
def classify_gene(row):
    if row['padj'] < P_CUTOFF and row['log2FoldChange'] > FC_CUTOFF:
        return 'Up-regulated'
    elif row['padj'] < P_CUTOFF and row['log2FoldChange'] < -FC_CUTOFF:
        return 'Down-regulated'
    else:
        return 'Not significant'

diff_result['gene_type'] = diff_result.apply(classify_gene, axis=1)

fig, ax = plt.subplots(figsize=(10, 8))
colors = {'Not significant': 'lightgray', 'Up-regulated': '#e64343', 'Down-regulated': '#436eee'}
for gene_type, color in colors.items():
    mask = diff_result['gene_type'] == gene_type
    ax.scatter(
        diff_result.loc[mask, 'log2FoldChange'],
        -np.log10(diff_result.loc[mask, 'padj']),
        c=color, s=15, alpha=0.7, label=gene_type
    )

# 添加阈值线
ax.axvline(x=FC_CUTOFF, color='black', linestyle='--', lw=1)
ax.axvline(x=-FC_CUTOFF, color='black', linestyle='--', lw=1)
ax.axhline(y=-np.log10(P_CUTOFF), color='black', linestyle='--', lw=1)

ax.set_xlabel('log₂(Fold Change)', fontweight='bold', fontsize=12)
ax.set_ylabel('-log₁₀(Adjusted P-value)', fontweight='bold', fontsize=12)
ax.set_title(f'{GSE_ID} Volcano Plot (DESeq2)\n{CASE_GROUP} vs {CONTROL_GROUP}', 
             fontweight='bold', fontsize=14, pad=20)
ax.legend(loc='upper right', frameon=True)
ax.set_xlim(-6, 6)

volcano_path = os.path.join(SAVE_DIR, f"{GSE_ID}_DESeq2_火山图.png")
plt.tight_layout()
plt.savefig(volcano_path, dpi=300, bbox_inches='tight')
plt.close()
print(f"✅ 火山图已保存：{volcano_path}")

# 统计上下调基因数量
up_count = (diff_result['gene_type'] == 'Up-regulated').sum()
down_count = (diff_result['gene_type'] == 'Down-regulated').sum()
print(f"   Up-regulated: {up_count}, Down-regulated: {down_count}")

# ==============================================================================
# 三、热图（Top N 差异基因，标题英文）
# ==============================================================================
print("\n" + "="*50)
print("开始绘制基因表达热图...")
print("="*50)

# 1. 按 padj 排序，取前 TOP_N_HEATMAP 个基因（显著且差异大）
diff_sorted = diff_result.sort_values('padj')
# 去重（按 Symbol，避免同一基因多行）
diff_sorted = diff_sorted.drop_duplicates(subset='Symbol', keep='first')
top_genes_info = diff_sorted.head(TOP_N_HEATMAP)
top_gene_ids = top_genes_info['id'].tolist()          # Ensembl ID
top_gene_symbols = top_genes_info['Symbol'].tolist()  # 用于标签

# 2. 从归一化表达矩阵中提取这些基因的数据（按 ID 匹配）
expr_sub = expr_norm.loc[expr_norm.index.isin(top_gene_ids)].copy()
# 按 top_gene_ids 顺序重排行
expr_sub = expr_sub.reindex(top_gene_ids)
# 将行名替换为 Symbol（方便热图阅读）
expr_sub.index = top_gene_symbols

print(f"✅ 有效热图基因数：{len(expr_sub)}")

if len(expr_sub) == 0:
    print("⚠️ 无有效差异基因，跳过热图绘制")
else:
    # 3. Z-score 标准化（按行）
    expr_z = expr_sub.sub(expr_sub.mean(axis=1), axis=0)
    expr_z = expr_z.div(expr_z.std(axis=1), axis=0).fillna(0)

    # 4. 样本标签简化（去掉 '_count' 后缀）
    sample_labels = [col.replace('_count', '') for col in expr_z.columns]

    # 5. 绘制热图
    cmap = LinearSegmentedColormap.from_list('custom_cmap', ['#3b76af', 'white', '#e37a57'])
    fig_height = max(8, len(expr_sub) * 0.3 + 2)
    fig, ax = plt.subplots(figsize=(12, fig_height))
    sns.heatmap(
        expr_z,
        cmap=cmap,
        center=0,
        linewidths=0.5,
        xticklabels=sample_labels,
        yticklabels=expr_z.index,
        cbar_kws={'label': 'Z-score', 'shrink': 0.8},
        ax=ax
    )

    ax.set_title(f'{GSE_ID} Top {len(expr_sub)} Differentially Expressed Genes (DESeq2)\n{CASE_GROUP} vs {CONTROL_GROUP}',
                 fontweight='bold', fontsize=16, pad=20)

    plt.xticks(rotation=45, ha='right', fontsize=10)
    plt.yticks(fontsize=10)

    heatmap_path = os.path.join(SAVE_DIR, f"{GSE_ID}_DESeq2_基因热图.png")
    plt.tight_layout()
    plt.savefig(heatmap_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"✅ 热图已保存：{heatmap_path}")

print("\n🎉 All plotting completed!")
print(f"📊 Files saved in: {SAVE_DIR}")
print("Output files:")
print(f"  - {os.path.basename(volcano_path)}")
print(f"  - {os.path.basename(heatmap_path)}")



### GSE85051



In [ ]:


import pandas as pd
import numpy as np
import os
from pathlib import Path
import gzip
import requests

# ===================== 统一预处理类：适配GSE85051 系列矩阵文件（基因水平输出） =====================
class GSE85051_UnifiedPreprocessor:
    def __init__(self, raw_data_dir, gpl_file=None):
        self.raw_dir = Path(raw_data_dir)
        self.gpl_file = Path(gpl_file) if gpl_file else None
        # ================== 完整的样本分组映射（60个样本）==================
        self.sample_mapping = {
            "GSM2257143": {"group": "Contralateral", "tissue": "LymphNode", "sample_name": "Contralateral_LymphNode_1"},
            "GSM2257144": {"group": "Contralateral", "tissue": "LymphNode", "sample_name": "Contralateral_LymphNode_2"},
            "GSM2257145": {"group": "Contralateral", "tissue": "LymphNode", "sample_name": "Contralateral_LymphNode_3"},
            "GSM2257146": {"group": "Contralateral", "tissue": "LymphNode", "sample_name": "Contralateral_LymphNode_4"},
            "GSM2257147": {"group": "Contralateral", "tissue": "LymphNode", "sample_name": "Contralateral_LymphNode_5"},
            "GSM2257148": {"group": "Contralateral", "tissue": "LymphNode", "sample_name": "Contralateral_LymphNode_6"},
            "GSM2257149": {"group": "Contralateral", "tissue": "LymphNode", "sample_name": "Contralateral_LymphNode_7"},
            "GSM2257150": {"group": "Contralateral", "tissue": "LymphNode", "sample_name": "Contralateral_LymphNode_8"},
            "GSM2257151": {"group": "Contralateral", "tissue": "LymphNode", "sample_name": "Contralateral_LymphNode_9"},
            "GSM2257152": {"group": "Contralateral", "tissue": "LymphNode", "sample_name": "Contralateral_LymphNode_10"},
            "GSM2257153": {"group": "Contralateral", "tissue": "LymphNode", "sample_name": "Contralateral_LymphNode_11"},
            "GSM2257154": {"group": "Injured", "tissue": "LymphNode", "sample_name": "Injured_LymphNode_1"},
            "GSM2257155": {"group": "Injured", "tissue": "LymphNode", "sample_name": "Injured_LymphNode_2"},
            "GSM2257156": {"group": "Injured", "tissue": "LymphNode", "sample_name": "Injured_LymphNode_3"},
            "GSM2257157": {"group": "Injured", "tissue": "LymphNode", "sample_name": "Injured_LymphNode_4"},
            "GSM2257158": {"group": "Injured", "tissue": "LymphNode", "sample_name": "Injured_LymphNode_5"},
            "GSM2257159": {"group": "Injured", "tissue": "LymphNode", "sample_name": "Injured_LymphNode_6"},
            "GSM2257160": {"group": "Injured", "tissue": "LymphNode", "sample_name": "Injured_LymphNode_7"},
            "GSM2257161": {"group": "Injured", "tissue": "LymphNode", "sample_name": "Injured_LymphNode_8"},
            "GSM2257162": {"group": "Injured", "tissue": "LymphNode", "sample_name": "Injured_LymphNode_9"},
            "GSM2257163": {"group": "Injured", "tissue": "LymphNode", "sample_name": "Injured_LymphNode_10"},
            "GSM2257164": {"group": "Injured", "tissue": "LymphNode", "sample_name": "Injured_LymphNode_11"},
            "GSM2257165": {"group": "Contralateral", "tissue": "SynovialTissue", "sample_name": "Contralateral_Synovial_1"},
            "GSM2257166": {"group": "Contralateral", "tissue": "SynovialTissue", "sample_name": "Contralateral_Synovial_2"},
            "GSM2257167": {"group": "Contralateral", "tissue": "SynovialTissue", "sample_name": "Contralateral_Synovial_3"},
            "GSM2257168": {"group": "Contralateral", "tissue": "SynovialTissue", "sample_name": "Contralateral_Synovial_4"},
            "GSM2257169": {"group": "Contralateral", "tissue": "SynovialTissue", "sample_name": "Contralateral_Synovial_5"},
            "GSM2257170": {"group": "Contralateral", "tissue": "SynovialTissue", "sample_name": "Contralateral_Synovial_6"},
            "GSM2257171": {"group": "Contralateral", "tissue": "SynovialTissue", "sample_name": "Contralateral_Synovial_7"},
            "GSM2257172": {"group": "Contralateral", "tissue": "SynovialTissue", "sample_name": "Contralateral_Synovial_8"},
            "GSM2257173": {"group": "Contralateral", "tissue": "SynovialTissue", "sample_name": "Contralateral_Synovial_9"},
            "GSM2257174": {"group": "Contralateral", "tissue": "SynovialTissue", "sample_name": "Contralateral_Synovial_10"},
            "GSM2257175": {"group": "Contralateral", "tissue": "SynovialTissue", "sample_name": "Contralateral_Synovial_11"},
            "GSM2257176": {"group": "Injured", "tissue": "SynovialTissue", "sample_name": "Injured_Synovial_1"},
            "GSM2257177": {"group": "Injured", "tissue": "SynovialTissue", "sample_name": "Injured_Synovial_2"},
            "GSM2257178": {"group": "Injured", "tissue": "SynovialTissue", "sample_name": "Injured_Synovial_3"},
            "GSM2257179": {"group": "Injured", "tissue": "SynovialTissue", "sample_name": "Injured_Synovial_4"},
            "GSM2257180": {"group": "Injured", "tissue": "SynovialTissue", "sample_name": "Injured_Synovial_5"},
            "GSM2257181": {"group": "Injured", "tissue": "SynovialTissue", "sample_name": "Injured_Synovial_6"},
            "GSM2257182": {"group": "Injured", "tissue": "SynovialTissue", "sample_name": "Injured_Synovial_7"},
            "GSM2257183": {"group": "Injured", "tissue": "SynovialTissue", "sample_name": "Injured_Synovial_8"},
            "GSM2257184": {"group": "Injured", "tissue": "SynovialTissue", "sample_name": "Injured_Synovial_9"},
            "GSM2257185": {"group": "Injured", "tissue": "SynovialTissue", "sample_name": "Injured_Synovial_10"},
            "GSM2257186": {"group": "Injured", "tissue": "SynovialTissue", "sample_name": "Injured_Synovial_11"},
            "GSM2257187": {"group": "Control", "tissue": "LymphNode", "sample_name": "Control_LymphNode_1"},
            "GSM2257188": {"group": "Control", "tissue": "LymphNode", "sample_name": "Control_LymphNode_2"},
            "GSM2257189": {"group": "Control", "tissue": "LymphNode", "sample_name": "Control_LymphNode_3"},
            "GSM2257190": {"group": "Control", "tissue": "LymphNode", "sample_name": "Control_LymphNode_4"},
            "GSM2257191": {"group": "Control", "tissue": "LymphNode", "sample_name": "Control_LymphNode_5"},
            "GSM2257192": {"group": "Control", "tissue": "LymphNode", "sample_name": "Control_LymphNode_6"},
            "GSM2257193": {"group": "Control", "tissue": "LymphNode", "sample_name": "Control_LymphNode_7"},
            "GSM2257194": {"group": "Control", "tissue": "LymphNode", "sample_name": "Control_LymphNode_8"},
            "GSM2257195": {"group": "Control", "tissue": "LymphNode", "sample_name": "Control_LymphNode_9"},
            "GSM2257196": {"group": "Control", "tissue": "SynovialTissue", "sample_name": "Control_Synovial_1"},
            "GSM2257197": {"group": "Control", "tissue": "SynovialTissue", "sample_name": "Control_Synovial_2"},
            "GSM2257198": {"group": "Control", "tissue": "SynovialTissue", "sample_name": "Control_Synovial_3"},
            "GSM2257199": {"group": "Control", "tissue": "SynovialTissue", "sample_name": "Control_Synovial_4"},
            "GSM2257200": {"group": "Control", "tissue": "SynovialTissue", "sample_name": "Control_Synovial_5"},
            "GSM2257201": {"group": "Control", "tissue": "SynovialTissue", "sample_name": "Control_Synovial_6"},
            "GSM2257202": {"group": "Control", "tissue": "SynovialTissue", "sample_name": "Control_Synovial_7"}
        }

    def read_series_matrix(self):
        """读取 GEO 系列矩阵文件，返回探针水平表达矩阵（已数值化，列名为 GSM ID）"""
        matrix_path = self.raw_dir / "GSE85051_series_matrix.txt.gz"
        if not matrix_path.exists():
            raise FileNotFoundError(f"未找到核心文件：{matrix_path.name}")

        df = pd.read_csv(matrix_path, sep="\t", compression="gzip", comment="!", header=0)
        df = df.set_index(df.columns[0])  # 第一列为探针 ID
        df = df.apply(pd.to_numeric, errors="coerce")
        df = df.dropna(how='all')

        new_cols = []
        for col in df.columns:
            if isinstance(col, str) and "." in col:
                new_cols.append(col.split(".")[0])
            else:
                new_cols.append(str(col))
        df.columns = new_cols
        return df

    def load_gpl_mapping(self):
        """加载平台文件，返回探针到基因符号的映射字典（探针ID -> 基因符号）"""
        if self.gpl_file and self.gpl_file.exists():
            gpl_path = self.gpl_file
        else:
            candidates = [
                self.raw_dir / "GPL17117-26578.txt",
                self.raw_dir / "GSE85051_RAW" / "GPL17117-26578.txt",
                self.raw_dir / "GPL16686.txt",
                self.raw_dir / "GPL16686-26578.txt"
            ]
            gpl_path = None
            for cand in candidates:
                if cand.exists():
                    gpl_path = cand
                    break
            if gpl_path is None:
                raise FileNotFoundError("未找到 GPL 文件...")

        print(f"使用平台文件：{gpl_path}")
        gpl = pd.read_csv(gpl_path, sep="\t", comment="#", header=0, low_memory=False, dtype=str)
        # 寻找探针ID列
        id_col = None
        for col in gpl.columns:
            if col.upper() == 'ID':
                id_col = col
                break
        if id_col is None:
            id_col = gpl.columns[0]
        # 寻找 gene_assignment 列
        gene_col = None
        for col in gpl.columns:
            if 'gene_assignment' in col.lower():
                gene_col = col
                break
        if gene_col is None:
            raise ValueError("未找到 'gene_assignment' 列")

        mapping = {}
        for idx, row in gpl.iterrows():
            probe = str(row[id_col]).strip()
            gene_field = row[gene_col]
            if pd.isna(gene_field) or gene_field == '---':
                continue
            first_assignment = str(gene_field).split('///')[0].strip()
            parts = [p.strip() for p in first_assignment.split('//')]
            gene_symbol = parts[1] if len(parts) >= 2 else parts[0]
            if gene_symbol and gene_symbol not in ('---', '', 'NA', 'nan'):
                gene_symbol = gene_symbol.split()[0]
                mapping[probe] = gene_symbol
        print(f"✅ 成功加载探针到基因映射：{len(mapping)} 个探针对应非空基因符号")
        return mapping

    def run(self):
        print("🔍 读取 GSE85051 官方系列矩阵文件...")
        exp_matrix = self.read_series_matrix()
        print(f"✅ 原始矩阵：{exp_matrix.shape[0]} 探针 × {exp_matrix.shape[1]} 样本")

        # 筛选有效样本
        valid_gsm = [gsm for gsm in exp_matrix.columns if gsm in self.sample_mapping]
        if len(valid_gsm) == 0:
            raise ValueError("未找到任何匹配的样本，请检查 sample_mapping 是否包含正确的 GSM ID")
        exp_filtered_samples = exp_matrix[valid_gsm]
        print(f"✅ 保留 {len(valid_gsm)} 个样本")

        # 检查表达矩阵是否全为 NaN
        if exp_filtered_samples.empty or exp_filtered_samples.isnull().all().all():
            raise ValueError("表达矩阵无有效数值，请检查数据文件或样本映射")

        # 检查 log2 转换
        data_vals = exp_filtered_samples.values.flatten()
        data_vals = data_vals[~np.isnan(data_vals)]
        if data_vals.size == 0:
            raise ValueError("表达矩阵所有值为 NaN，无法继续")
        max_val = np.max(data_vals)
        min_val = np.min(data_vals)
        print(f"📊 表达值范围：min={min_val:.2f}, max={max_val:.2f}")
        if max_val > 50:
            need_log2 = True
            print("⚠️ 未 log2 转换，将执行 log2(expression+1)")
        else:
            need_log2 = False
            print("✅ 已 log2 转换，不再重复")

        # 低表达过滤
        min_exp_for_filter = 1.0
        min_samples = max(3, int(exp_filtered_samples.shape[1] * 0.1))
        keep_rows = (exp_filtered_samples > min_exp_for_filter).sum(axis=1) >= min_samples
        exp_filtered = exp_filtered_samples.loc[keep_rows]
        print(f"✅ 低表达过滤后保留 {exp_filtered.shape[0]} 个探针")

        # 标准化
        if need_log2:
            exp_final_probe = np.log2(exp_filtered + 1)
        else:
            exp_final_probe = exp_filtered

        # 关键修复：将探针索引转换为字符串
        exp_final_probe.index = exp_final_probe.index.astype(str)

        # 探针到基因映射
        print("\n🔍 加载平台文件，进行探针到基因的映射...")
        probe_to_gene = self.load_gpl_mapping()

        common_probes = [p for p in exp_final_probe.index if p in probe_to_gene]
        print(f"🔍 表达矩阵中的探针总数：{len(exp_final_probe.index)}")
        print(f"🔍 映射字典中的探针数：{len(probe_to_gene)}")
        print(f"🔍 成功匹配的探针数：{len(common_probes)}")
        if len(common_probes) == 0:
            print("⚠️ 警告：没有匹配到任何探针，请检查探针 ID 格式是否一致")
            print("示例探针（前5个）：", list(exp_final_probe.index[:5]))
            print("示例映射键（前5个）：", list(probe_to_gene.keys())[:5])
            exp_gene = pd.DataFrame(index=[], columns=exp_final_probe.columns)
        else:
            exp_mapped = exp_final_probe.loc[common_probes].copy()
            exp_mapped['GeneSymbol'] = [probe_to_gene[p] for p in exp_mapped.index]
            exp_gene = exp_mapped.groupby('GeneSymbol').mean()
            exp_gene = exp_gene.drop(columns=['GeneSymbol'], errors='ignore')
            print(f"✅ 基因水平表达矩阵：{exp_gene.shape[0]} 基因 × {exp_gene.shape[1]} 样本")

        # 重命名样本列
        rename_dict = {g: self.sample_mapping[g]['sample_name'] for g in valid_gsm}
        exp_gene = exp_gene.rename(columns=rename_dict)

        # 样本信息表
        sample_info = pd.DataFrame([
            {"sample_name": self.sample_mapping[g]['sample_name'],
             "group": self.sample_mapping[g]['group'],
             "tissue": self.sample_mapping[g]['tissue'],
             "gsm_id": g} for g in valid_gsm
        ])

        # 保存
        output_dir = Path("GSE85051_final")
        output_dir.mkdir(exist_ok=True)
        exp_gene.to_csv(output_dir / "gene_expression_matrix.csv")
        sample_info.to_csv(output_dir / "sample_info.csv", index=False)
        exp_final_probe.to_csv(output_dir / "probe_expression_matrix.csv")

        print(f"💾 数据已保存至 {output_dir}")
        return exp_gene, sample_info


# ===================== 一键执行 =====================
if __name__ == "__main__":
    RAW_DATA_FOLDER = "./17 骨关节炎和囊泡/GEO 血友病滑膜 vs. 骨关节炎滑膜的差异基因/标本为鼠"
    GPL_FILE_PATH = "./17 骨关节炎和囊泡/GEO 血友病滑膜 vs. 骨关节炎滑膜的差异基因/标本为鼠/GSE85051_RAW/GPL17117-26578.txt"

    processor = GSE85051_UnifiedPreprocessor(RAW_DATA_FOLDER, gpl_file=GPL_FILE_PATH)
    GSE85051_gene_expr, sample_info = processor.run()

    print("\n" + "="*60)
    print("📊 GSE85051 预处理完成！")
    print(f"📦 最终数据维度：{GSE85051_gene_expr.shape[0]} 基因 × {GSE85051_gene_expr.shape[1]} 样本")
    print("\n🔍 表达矩阵预览（前5行×5列）：")
    if GSE85051_gene_expr.shape[0] > 0:
        print(GSE85051_gene_expr.iloc[:5, :5].round(2))
    else:
        print("空矩阵（无基因）")
    print("\n📄 样本分组统计：")
    print(sample_info.groupby(["group", "tissue"]).size())




In [ ]:

    
# ==============================================
# GSE85051 差异分析（limma 双因素交互模型）
# 修正：参考组列名缺失问题 + 表达矩阵维度
# ==============================================
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap

# rpy2 相关
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri, StrVector
from rpy2.robjects.conversion import localconverter
from rpy2.robjects.packages import importr, isinstalled

# 安装/加载 limma
utils = importr('utils')
if not isinstalled('limma'):
    print("正在安装 limma ...")
    utils.install_packages('BiocManager', quiet=True)
    ro.r('BiocManager::install("limma", update=FALSE, ask=FALSE)')
limma = importr('limma')
stats = importr('stats')

# ===================== 路径配置 =====================
OUTPUT_DIR = Path("GSE85051_final")
EXPR_FILE = OUTPUT_DIR / "gene_expression_matrix.csv"
SAMPLE_FILE = OUTPUT_DIR / "sample_info.csv"
RESULT_DIR = Path("GSE85051_diff_results")
RESULT_DIR.mkdir(exist_ok=True)

# ===================== 1. 读取数据 =====================
print("="*50)
print("读取表达矩阵和样本信息...")
print("="*50)
expr_df = pd.read_csv(EXPR_FILE, index_col=0)          # 基因 × 样本（正确格式，无需转置）
sample_info = pd.read_csv(SAMPLE_FILE)                 # sample_name, group, tissue
print(f"✅ 表达矩阵：{expr_df.shape[0]} 基因 × {expr_df.shape[1]} 样本")
print(f"✅ 样本信息：{sample_info.shape[0]} 个样本")

# 确保样本顺序一致（关键：表达矩阵列 = 样本信息行）
sample_info = sample_info.set_index('sample_name').loc[expr_df.columns].reset_index()
print("\n样本分组统计：")
print(sample_info.groupby(['group', 'tissue']).size())

# ===================== 2. 准备数据并转换到 R =====================
group_vector = StrVector(sample_info['group'].tolist())
tissue_vector = StrVector(sample_info['tissue'].tolist())

# 直接使用 基因×样本 的原始矩阵
with localconverter(ro.default_converter + pandas2ri.converter):
    expr_r = ro.conversion.py2rpy(expr_df)

ro.globalenv['expr_mat'] = ro.r('as.matrix')(expr_r)
ro.globalenv['group'] = group_vector
ro.globalenv['tissue'] = tissue_vector

# ===================== 3. 运行 limma 线性模型 =====================
ro.r('''
    library(limma)
    group <- factor(group)
    tissue <- factor(tissue)
    # 打印因子水平，确认参考组
    cat("group 因子水平：\n"); print(levels(group))
    cat("tissue 因子水平：\n"); print(levels(tissue))
    
    design <- model.matrix(~ group * tissue)
    colnames(design) <- make.names(colnames(design))
    # 打印设计矩阵列名（核心调试）
    cat("\n设计矩阵列名：\n"); print(colnames(design))
    
    fit <- lmFit(expr_mat, design)
    fit <- eBayes(fit)
''')

# ===================== 4. 构建对比矩阵=====================
# 修正原因：Contralateral 是参考组，无groupContralateral列，系数=0
ro.r('''
    cont.matrix <- makeContrasts(
        # 淋巴结 (LN) 对比：Contralateral为参考组，直接用0替代
        Injured_vs_Control_LN = groupInjured - groupControl,
        Contralateral_vs_Control_LN = 0 - groupControl,
        
        # 滑膜组织 (ST) 对比
        Injured_vs_Control_ST = (groupInjured + groupInjured.tissueSynovialTissue) - (groupControl + groupControl.tissueSynovialTissue),
        Contralateral_vs_Control_ST = 0 - (groupControl + groupControl.tissueSynovialTissue),
        
        # 交互效应
        Interaction_Injured_STvsLN = groupInjured.tissueSynovialTissue - groupControl.tissueSynovialTissue,
        
        levels = design
    )
    fit2 <- contrasts.fit(fit, cont.matrix)
    fit2 <- eBayes(fit2)
''')

# ===================== 5. 提取结果函数 =====================
def get_results(contrast_name):
    with localconverter(ro.default_converter + pandas2ri.converter):
        ro.globalenv['contrast'] = contrast_name
        ro.r(f'''
            res <- topTable(fit2, coef="{contrast_name}", number=Inf, sort.by="none")
            res_df <- as.data.frame(res)
        ''')
        res_pd = ro.globalenv['res_df']
    res_pd = res_pd.reset_index().rename(columns={'index': 'Gene'})
    return res_pd

contrasts_list = [
    "Injured_vs_Control_LN",
    "Contralateral_vs_Control_LN",
    "Injured_vs_Control_ST",
    "Contralateral_vs_Control_ST",
    "Interaction_Injured_STvsLN"
]

for cname in contrasts_list:
    print(f"正在提取对比：{cname}")
    df_res = get_results(cname)
    df_res.to_csv(RESULT_DIR / f"{cname}_results.csv", index=False)
    sig = (df_res['adj.P.Val'] < 0.05) & (abs(df_res['logFC']) > 1)
    up = df_res[sig & (df_res['logFC'] > 0)]
    down = df_res[sig & (df_res['logFC'] < 0)]
    print(f"  显著上调: {len(up)}, 显著下调: {len(down)}")
    up.to_csv(RESULT_DIR / f"{cname}_upregulated.csv", index=False)
    down.to_csv(RESULT_DIR / f"{cname}_downregulated.csv", index=False)

# ===================== 6. 火山图（以 Injured_vs_Control_LN 为例）=====================
print("\n" + "="*50)
print("绘制火山图...")
print("="*50)
res_ln = pd.read_csv(RESULT_DIR / "Injured_vs_Control_LN_results.csv")
res_ln['-log10(P.Value)'] = -np.log10(res_ln['P.Value'])
res_ln['Sig'] = 'Not significant'
res_ln.loc[(res_ln['adj.P.Val'] < 0.05) & (res_ln['logFC'] > 1), 'Sig'] = 'Up-regulated'
res_ln.loc[(res_ln['adj.P.Val'] < 0.05) & (res_ln['logFC'] < -1), 'Sig'] = 'Down-regulated'

fig, ax = plt.subplots(figsize=(10, 8))
colors = {'Up-regulated': '#e64343', 'Down-regulated': '#436eee', 'Not significant': 'lightgray'}
for s, c in colors.items():
    subset = res_ln[res_ln['Sig'] == s]
    ax.scatter(subset['logFC'], subset['-log10(P.Value)'], c=c, s=15, alpha=0.7, label=s)
ax.axvline(x=1, linestyle='--', color='black', lw=1)
ax.axvline(x=-1, linestyle='--', color='black', lw=1)
ax.axhline(y=-np.log10(0.05), linestyle='--', color='black', lw=1)
ax.set_xlabel('log₂(倍数变化)', fontsize=12)
ax.set_ylabel('-log₁₀(P值)', fontsize=12)
ax.set_title('GSE85051 火山图 (Injured vs Control, LymphNode)', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig(RESULT_DIR / "Volcano_Injured_vs_Control_LN.png", dpi=300)
plt.close()
print("✅ 火山图已保存")

# ===================== 7. 热图（Top 50 差异基因）=====================
print("\n" + "="*50)
print("绘制热图...")
print("="*50)
top_genes = res_ln.sort_values('adj.P.Val').head(50)['Gene'].tolist()
heat_data = expr_df.loc[expr_df.index.isin(top_genes), :].copy()
heat_data = heat_data.reindex(top_genes)
# Z-score 标准化
heat_z = heat_data.sub(heat_data.mean(axis=1), axis=0)
heat_z = heat_z.div(heat_data.std(axis=1), axis=0).fillna(0)
# 绘制
cmap = LinearSegmentedColormap.from_list('custom', ['#3b76af', 'white', '#e37a57'])
fig_height = max(10, len(top_genes)*0.3)
fig, ax = plt.subplots(figsize=(14, fig_height))
sns.heatmap(heat_z, cmap=cmap, center=0, xticklabels=True, yticklabels=True,
            cbar_kws={'label': 'Z-score'}, ax=ax)
ax.set_title('GSE85051 Top 50 差异基因 (Injured vs Control, LymphNode)', fontweight='bold')
plt.xticks(rotation=90, fontsize=8)
plt.yticks(fontsize=8)
plt.tight_layout()
plt.savefig(RESULT_DIR / "Heatmap_Top50_Injured_LN.png", dpi=300)
plt.close()
print("✅ 热图已保存")

print("\n🎉 差异分析及可视化全部完成！")
print(f"📁 结果保存至: {RESULT_DIR}")





### 小鼠关键基因和韦恩图



In [ ]:


# ==============================================
# 两个小鼠数据集差异基因交集分析与韦恩图
# 输入：
#   GSE234863 (DESeq2) 上调/下调基因CSV
#   GSE85051 (limma) 上调/下调基因CSV
# 输出：
#   每个数据集的差异基因并集CSV，交集CSV，韦恩图
# ==============================================
import pandas as pd
import numpy as np
import os
from pathlib import Path
import matplotlib.pyplot as plt
from matplotlib_venn import venn2

# ===================== 路径配置 =====================
# GSE234863 结果路径（SAVE_DIR 来自原始代码）
GSE234863_DIR = Path("./data")
# GSE85051 差异结果路径（RESULT_DIR 来自原始代码）
GSE85051_DIR = Path("GSE85051_diff_results")   # 若不在当前目录，请修改为绝对路径

# 输出目录
OUTPUT_DIR = Path("mouse_common_genes")
OUTPUT_DIR.mkdir(exist_ok=True)

# ===================== 1. 读取 GSE234863 差异基因 =====================
print("="*50)
print("读取 GSE234863 差异基因...")
print("="*50)

up_234863 = pd.read_csv(GSE234863_DIR / "GSE234863_DESeq2_upregulated.csv")
down_234863 = pd.read_csv(GSE234863_DIR / "GSE234863_DESeq2_downregulated.csv")

# 提取基因符号（Symbol列）
genes_up_234863 = set(up_234863["Symbol"].dropna().astype(str))
genes_down_234863 = set(down_234863["Symbol"].dropna().astype(str))
genes_all_234863 = genes_up_234863.union(genes_down_234863)
print(f"✅ GSE234863 上调基因数: {len(genes_up_234863)}")
print(f"✅ GSE234863 下调基因数: {len(genes_down_234863)}")
print(f"✅ GSE234863 总差异基因数（并集）: {len(genes_all_234863)}")

# 保存为 CSV
pd.DataFrame({"Gene": sorted(genes_all_234863)}).to_csv(
    OUTPUT_DIR / "GSE234863_all_diff_genes.csv", index=False
)

# ===================== 2. 读取 GSE85051 差异基因 =====================
print("\n" + "="*50)
print("读取 GSE85051 差异基因...")
print("="*50)

# 注意：GSE85051 对比中我们以 Injured_vs_Control_LN 为例，也可以汇总所有对比的差异基因。
# 为了得到全面的差异基因，建议取所有对比中显著的基因并集。
# 但根据常见需求，通常取某主要对比（如 Injured_vs_Control_LN）的差异基因。
# 此处以 Injured_vs_Control_LN 为例；如需合并多个对比，可扩展。

contrast_name = "Injured_vs_Control_LN"
up_85051 = pd.read_csv(GSE85051_DIR / f"{contrast_name}_upregulated.csv")
down_85051 = pd.read_csv(GSE85051_DIR / f"{contrast_name}_downregulated.csv")

genes_up_85051 = set(up_85051["Gene"].dropna().astype(str))
genes_down_85051 = set(down_85051["Gene"].dropna().astype(str))
genes_all_85051 = genes_up_85051.union(genes_down_85051)
print(f"✅ GSE85051 ({contrast_name}) 上调基因数: {len(genes_up_85051)}")
print(f"✅ GSE85051 ({contrast_name}) 下调基因数: {len(genes_down_85051)}")
print(f"✅ GSE85051 ({contrast_name}) 总差异基因数（并集）: {len(genes_all_85051)}")

pd.DataFrame({"Gene": sorted(genes_all_85051)}).to_csv(
    OUTPUT_DIR / "GSE85051_all_diff_genes.csv", index=False
)

# ===================== 3. 取两个数据集交集 =====================
common_genes = genes_all_234863.intersection(genes_all_85051)
print(f"\n✅ 两个数据集共同差异基因数: {len(common_genes)}")

pd.DataFrame({"Gene": sorted(common_genes)}).to_csv(
    OUTPUT_DIR / "common_diff_genes_between_GSE234863_and_GSE85051.csv", index=False
)

# ===================== 4. 绘制韦恩图 =====================
plt.figure(figsize=(8, 6))
venn2(subsets=(len(genes_all_234863 - common_genes),
               len(genes_all_85051 - common_genes),
               len(common_genes)),
      set_labels=('GSE234863', 'GSE85051'))
plt.title('Mouse Differential Expressed Genes Overlap')
plt.savefig(OUTPUT_DIR / "Venn_diagram_diff_genes.png", dpi=300, bbox_inches='tight')
plt.show()
print(f"✅ 韦恩图已保存至 {OUTPUT_DIR / 'Venn_diagram_diff_genes.png'}")

print("\n🎉 分析完成！所有输出文件保存在：", OUTPUT_DIR)